# Kali Linux Anomaly Detection – ML Notebook

**Project:** AI-Based System Resource Anomaly Detection and Performance Bottleneck Identification in Kali Linux

**Team:** `<names>` · **Seat no. / PRN:** `<numbers>` · **Course, date:** `<course, date>` · **Repository:** `<link>`

Models: Isolation Forest (primary), Local Outlier Factor and a percentile rule (non-ML baseline). Run in Colab with *Runtime → Run all* and upload `system.csv`, `experiments.csv` (optionally `process.csv`). Cells marked **WRITE** are for your own words, from your own results.

## 1. Libraries

The API must pin the same scikit-learn, NumPy, pandas and joblib versions that trained the model.

In [ ]:
# Google Colab already includes every library used here. If you run locally and one is missing,
# install it in a separate cell, for example:  !pip install scikit-learn seaborn
import io
import json
import platform
import re
import shutil
import time
import warnings
import zipfile
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", "{:,.3f}".format)

RANDOM_STATE = 42                      # used wherever a random choice is made
np.random.seed(RANDOM_STATE)

# Library versions: backend/requirements.txt must pin the same scikit-learn, NumPy, pandas and joblib.
LIBRARY_VERSIONS = {
    "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
    "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "joblib": joblib.__version__,
    "matplotlib": matplotlib.__version__, "seaborn": sns.__version__,
}
for name, version in LIBRARY_VERSIONS.items():
    print(f"{name:13s} {version}")

## 2. Run settings

`QUICK_RUN = True` is a trial run only. Report numbers from `QUICK_RUN = False`.

In [ ]:
# ---- Run settings: change them here and nowhere else ---------------------------------------------
QUICK_RUN = False            # True = tiny grid for a trial run. Never report numbers from a quick run.
INTERVAL_S = 5               # agent sampling interval in seconds
GAP_THRESHOLD_S = 15         # a pause longer than three sampling intervals counts as a gap in the data
TRIM_S = 30                  # samples this close to the start/end of a run are ambiguous and never scored
MIN_RUN_COVERAGE = 34 / 36   # a stress run is void if fewer than 34 of its 36 expected samples were stored
LOF_MAX_TRAIN_ROWS = 20_000  # LOF stores its training rows, so they are capped (random subsample, fixed seed)
IF_MAX_TRAIN_ROWS = 100_000  # time/memory guard for Isolation Forest; set to None to switch it off
API_URL = ""                 # base URL of the deployed API; used only by the health check in section H
LABELLED_HOSTS = None        # None = every machine takes labels from experiments.csv; or a list such as ["kali-01"] when only those were stressed

# ---- Labels ---------------------------------------------------------------------------------------
STRESS_LABELS = ["cpu", "memory", "disk", "network"]      # scored as anomaly (1)
LABEL_ORDER = ["normal"] + STRESS_LABELS                  # normal is scored as 0; cooldown is never scored
LABEL_COLORS = {"normal": "#2a9d8f", "cpu": "#e76f51", "memory": "#8d5fd3", "disk": "#e9c46a",
                "network": "#277da1", "cooldown": "#bdbdbd", "background": "#9e9e9e"}

print("QUICK_RUN =", QUICK_RUN, "->", "SMALL trial grid (do not report)" if QUICK_RUN else "FULL tuning grid")

# Section A – Dataset (rubric A)

### A.1 Load the exported files

Upload `system.csv`, `experiments.csv` and optionally `process.csv` when the dialog opens (outside Colab: put them in `./data` or the current folder).

In [ ]:
# Folder to search first. Leave as None to search the usual places.
DATA_DIR = None                  # e.g. "/content/drive/MyDrive/kali_project/data" after mounting Drive
USE_COLAB_UPLOAD = True          # set to False if the files are already in Drive or in the working folder

if USE_COLAB_UPLOAD:
    try:
        from google.colab import files            # only exists inside Google Colab
        print("Choose system.csv, experiments.csv and (optionally) process.csv ...")
        files.upload()                            # Colab saves the chosen files in the working folder
    except ImportError:
        print("Not running in Colab: looking for the CSV files in ./data and the current folder.")

SEARCH_DIRS = ([Path(DATA_DIR)] if DATA_DIR else []) + [Path("."), Path("data"), Path("/content"), Path("/content/data")]


def find_table(stem):
    """Return the first existing <stem>.csv or <stem>.csv.gz in SEARCH_DIRS, or None."""
    for folder in SEARCH_DIRS:
        for suffix in (".csv", ".csv.gz"):
            candidate = folder / f"{stem}{suffix}"
            if candidate.exists():
                return candidate
    return None


system_path, experiments_path, process_path = find_table("system"), find_table("experiments"), find_table("process")
if system_path is None:
    raise FileNotFoundError("system.csv was not found. Upload it (or set DATA_DIR) and rerun this cell.")
if experiments_path is None:
    raise FileNotFoundError("experiments.csv was not found. Sections E to G need the labelled runs; "
                            "upload it next to system.csv and rerun this cell.")

raw_system = pd.read_csv(system_path)
raw_experiments = pd.read_csv(experiments_path)
raw_process = pd.read_csv(process_path) if process_path else None

print("system.csv      :", system_path, raw_system.shape)
print("experiments.csv :", experiments_path, raw_experiments.shape)
print("process.csv     :", process_path if process_path else "not provided (the process chart will be skipped)",
      raw_process.shape if raw_process is not None else "")

### A.2 Check the columns and parse the timestamps

In [ ]:
# Attribute definitions from the specification: (field, type, unit or range, source)
SYSTEM_FIELDS = [
    ("sample_id", "uuid", "identifier", "uuid4() in the agent"),
    ("host_id", "text", "identifier", "configuration"),
    ("ts", "timestamptz", "UTC", "datetime.now(timezone.utc)"),
    ("cpu_percent", "real", "% of all cores, 0 to 100", "psutil.cpu_percent(interval=None)"),
    ("cpu_count", "smallint", "logical cores, constant", "psutil.cpu_count()"),
    ("ram_total", "bigint", "bytes, constant", "virtual_memory().total"),
    ("ram_used", "bigint", "bytes", "virtual_memory().used"),
    ("ram_available", "bigint", "bytes", "virtual_memory().available"),
    ("ram_percent", "real", "%, 0 to 100", "virtual_memory().percent"),
    ("swap_percent", "real", "%, 0 to 100", "swap_memory().percent"),
    ("disk_used", "bigint", "bytes", 'disk_usage("/").used'),
    ("disk_free", "bigint", "bytes", 'disk_usage("/").free'),
    ("disk_percent", "real", "%, 0 to 100", 'disk_usage("/").percent'),
    ("disk_read_rate", "real, nullable", "bytes per second, 0 or more", "change in disk_io_counters().read_bytes / seconds"),
    ("disk_write_rate", "real, nullable", "bytes per second, 0 or more", "change in disk_io_counters().write_bytes / seconds"),
    ("network_sent_rate", "real, nullable", "bytes per second, 0 or more", "change in bytes_sent / seconds (all interfaces except lo)"),
    ("network_recv_rate", "real, nullable", "bytes per second, 0 or more", "change in bytes_recv / seconds (all interfaces except lo)"),
    ("load_average", "real", "1-minute load, 0 or more", "psutil.getloadavg()[0]"),
    ("process_count", "integer", "count", "len(psutil.pids())"),
]
PROCESS_FIELDS = [
    ("pid", "integer", "identifier", "pid"),
    ("process_name", "text (up to 64 characters)", "categorical", "name()"),
    ("cpu_percent", "real", "% of one core, can exceed 100 on multi-core", "cpu_percent()"),
    ("memory_percent", "real", "% of RAM, 0 to 100", "memory_percent()"),
    ("memory_mb", "real", "MiB resident", "memory_info().rss / 2^20"),
    ("num_threads", "integer", "count", "num_threads()"),
    ("status", "text", "categorical (running, sleeping, disk-sleep, ...)", "status()"),
    ("io_read_rate", "real, nullable", "bytes per second", "change in io_counters().read_bytes / seconds"),
    ("io_write_rate", "real, nullable", "bytes per second", "change in io_counters().write_bytes / seconds"),
]

RATE_FIELDS = ["disk_read_rate", "disk_write_rate", "network_sent_rate", "network_recv_rate"]
REQUIRED_SYSTEM_COLUMNS = ["ts", "cpu_percent", "cpu_count", "ram_percent", "swap_percent", "load_average",
                           "process_count"] + RATE_FIELDS

missing = [c for c in REQUIRED_SYSTEM_COLUMNS if c not in raw_system.columns]
if missing:
    raise ValueError(f"system.csv is missing required columns: {missing}. Rename them or re-export the data.")

# host_id and sample_id are optional here: create stand-ins so later steps work for single-machine files.
if "host_id" not in raw_system.columns:
    print("No host_id column: assuming one machine called 'host-1'.")
    raw_system["host_id"] = "host-1"
if "sample_id" not in raw_system.columns:
    print("No sample_id column: creating row identifiers.")
    raw_system["sample_id"] = [f"row-{i}" for i in range(len(raw_system))]

# Timestamps are UTC. Unparseable values become NaT and are dropped in section D, step 1.
raw_system["ts"] = pd.to_datetime(raw_system["ts"], utc=True, errors="coerce")
numeric_columns = [name for name, _, _, _ in SYSTEM_FIELDS if name in raw_system.columns
                   and name not in ("sample_id", "host_id", "ts")]
raw_system[numeric_columns] = raw_system[numeric_columns].apply(pd.to_numeric, errors="coerce")
print("Required columns present. Unparseable timestamps:", int(raw_system["ts"].isna().sum()))

### A.3 Overview: shape, column types, time span and gaps

In [ ]:
print("Rows x columns:", raw_system.shape)
display(raw_system.dtypes.to_frame("dtype").T)

# Time span, sampling coverage and gaps for each host
overview, gap_rows = [], []
for host, g in raw_system.groupby("host_id"):
    t = g["ts"].dropna().sort_values()
    span_s = (t.iloc[-1] - t.iloc[0]).total_seconds()
    step = t.diff().dt.total_seconds()
    gaps = step[step > GAP_THRESHOLD_S]
    overview.append({"host_id": host, "rows": len(t), "first sample": t.iloc[0], "last sample": t.iloc[-1],
                     "span (hours)": span_s / 3600, "expected rows": span_s / INTERVAL_S + 1,
                     "gaps": len(gaps), "time lost in gaps (min)": gaps.sum() / 60})
    for idx, seconds in gaps.items():
        gap_rows.append({"host_id": host, "gap starts after": t.loc[:idx].iloc[-2], "gap length (min)": seconds / 60})

overview = pd.DataFrame(overview)
display(overview)
if gap_rows:
    print("Longest gaps:")
    display(pd.DataFrame(gap_rows).sort_values("gap length (min)", ascending=False).head(10))
else:
    print("No gaps longer than", GAP_THRESHOLD_S, "seconds.")

### A.4 Labels and sessions

A sample gets the label of the run containing its timestamp. Cooldown, trimmed (within 30 s of a run edge), void (under 34 of 36 samples) and unlabelled rows are never scored. If `experiments.csv` has no `host_id` and only one machine was stressed, set `LABELLED_HOSTS`.

In [ ]:
# Read and clean the run table
required_runs = ["session_id", "label", "started_at", "ended_at"]
missing_runs = [c for c in required_runs if c not in raw_experiments.columns]
if missing_runs:
    raise ValueError(f"experiments.csv is missing columns: {missing_runs}")

runs = raw_experiments.copy()
runs["started_at"] = pd.to_datetime(runs["started_at"], utc=True, errors="coerce")
runs["ended_at"] = pd.to_datetime(runs["ended_at"], utc=True, errors="coerce")
runs["label"] = runs["label"].astype(str).str.strip().str.lower()
runs["session_id"] = runs["session_id"].astype(str).str.strip()
runs["intensity"] = runs["intensity"].fillna("").astype(str).str.strip().str.lower() if "intensity" in runs.columns else ""
if "run_id" not in runs.columns:
    runs["run_id"] = [f"run-{i + 1}" for i in range(len(runs))]

unreadable = runs["started_at"].isna() | runs["ended_at"].isna()
if unreadable.any():
    print("WARNING: dropping", int(unreadable.sum()), "runs with unreadable start or end times.")
runs = runs.loc[~unreadable].sort_values("started_at").reset_index(drop=True)
unknown = sorted(set(runs["label"]) - set(LABEL_ORDER + ["cooldown"]))
if unknown:
    print("WARNING: unknown labels (they are never scored and never used for training):", unknown)
if "host_id" not in runs.columns and raw_system["host_id"].nunique() > 1 and LABELLED_HOSTS is None:
    print(f"WARNING: experiments.csv has no host_id column but system.csv holds {raw_system['host_id'].nunique()} machines, so every machine's "
          "samples inside a run interval receive that run's label. If the runs were executed on one machine only, set "
          "LABELLED_HOSTS = ['<that host_id>'] in the run-settings cell; the other machines then count as unlabelled background.")

# One row per session, in time order. A session begins when its first run begins.
sessions = runs.groupby("session_id").agg(start=("started_at", "min"), end=("ended_at", "max"),
                                          runs=("run_id", "count")).sort_values("start")
sessions["order"] = range(1, len(sessions) + 1)


def attach_labels(df, runs_table):
    """Label every sample by the run whose interval contains its timestamp.

    Adds the columns run_id, session_id, label, intensity, is_trimmed, run_void and in_session_gap.
    Returns the labelled frame and a per-run summary (expected versus stored samples).
    """
    n = len(df)
    ts = df["ts"]
    hosts = df["host_id"].to_numpy()
    run_id = np.full(n, None, dtype=object)
    session_id = np.full(n, None, dtype=object)
    label = np.full(n, None, dtype=object)
    intensity = np.full(n, "", dtype=object)
    trimmed = np.zeros(n, dtype=bool)
    void = np.zeros(n, dtype=bool)
    n_matches = np.zeros(n, dtype=int)
    summary = []

    for r in runs_table.itertuples(index=False):
        # a run covers [start, end): a sample exactly on the boundary between two runs belongs to the later one
        inside = ((ts >= r.started_at) & (ts < r.ended_at)).to_numpy(dtype=bool, copy=True)   # writable copy
        run_host = getattr(r, "host_id", None)
        if run_host is not None and pd.notna(run_host):
            inside &= (hosts == run_host)                 # a run only labels the machine it was run on
        if LABELLED_HOSTS is not None:
            inside &= np.isin(hosts, LABELLED_HOSTS)      # only these machines were stressed; the others stay background
        n_matches += inside
        run_id[inside], session_id[inside] = r.run_id, r.session_id
        label[inside], intensity[inside] = r.label, r.intensity
        # samples close to either end of the run are ambiguous
        since_start = (ts[inside] - r.started_at).dt.total_seconds().to_numpy()
        until_end = (r.ended_at - ts[inside]).dt.total_seconds().to_numpy()
        trimmed[inside] = (since_start < TRIM_S) | (until_end < TRIM_S)
        # a run with too few stored samples is void
        expected = max(int(round((r.ended_at - r.started_at).total_seconds() / INTERVAL_S)), 1)
        stored = int(inside.sum())
        # only stress runs can be void: a normal block with a short outage still holds valid normal rows
        is_void = (r.label in STRESS_LABELS) and bool(stored < np.floor(expected * MIN_RUN_COVERAGE + 1e-6))
        void[inside] = is_void
        summary.append({"run_id": r.run_id, "session_id": r.session_id, "label": r.label, "intensity": r.intensity,
                        "expected_samples": expected, "stored_samples": stored, "void": bool(is_void)})

    # Unlabelled samples that fall inside a session are not ordinary background use: flag them.
    unlabelled = pd.isna(label)
    in_gap = np.zeros(n, dtype=bool)
    group_cols = ["session_id"] + (["host_id"] if "host_id" in runs_table.columns else [])
    for key, g in runs_table.groupby(group_cols):
        key = key if isinstance(key, tuple) else (key,)
        mask = unlabelled & ((ts >= g["started_at"].min()) & (ts < g["ended_at"].max())).to_numpy()
        if len(key) > 1:
            mask &= (hosts == key[1])
        in_gap |= mask

    out = df.copy()
    out["run_id"], out["session_id"], out["label"], out["intensity"] = run_id, session_id, label, intensity
    out["is_trimmed"], out["run_void"], out["in_session_gap"] = trimmed, void, in_gap
    if (n_matches > 1).any():
        print(f"WARNING: {int((n_matches > 1).sum())} samples lie inside more than one run (overlapping runs); the last run wins.")
    return out, pd.DataFrame(summary)


labelled_raw, run_summary_raw = attach_labels(raw_system, runs)

print("Runs:", len(runs), "| sessions:", len(sessions))
display(sessions)
void_runs = run_summary_raw[run_summary_raw["void"]]
print("Void runs (too few stored samples):", len(void_runs))
if len(void_runs):
    display(void_runs)
print("Unlabelled samples inside sessions:", int(labelled_raw["in_session_gap"].sum()))

### A.5 Rows per session and label

In [ ]:
# Count samples per session and label (raw, before trimming)
tmp = labelled_raw.assign(session=labelled_raw["session_id"].fillna("background"),
                          group=labelled_raw["label"].fillna("background"))
counts = pd.crosstab(tmp["session"], tmp["group"])
counts = counts[[c for c in ["background", "normal"] + STRESS_LABELS + ["cooldown"] if c in counts.columns]]
counts["total"] = counts.sum(axis=1)
display(counts)

print("Rows per host:")
display(labelled_raw["host_id"].value_counts().to_frame("rows").T)

### A.6 Attribute table – system metrics (specification 5.1)

In [ ]:
# Helper that joins the specification rows with what the data actually contains
def attribute_table(df, spec_rows):
    """Specification rows plus the observed range, missing values and distinct values."""
    rows = []
    for name, type_text, unit_text, source in spec_rows:
        if name not in df.columns:
            rows.append({"field": name, "type": type_text, "unit / range": unit_text, "source": source,
                         "observed range": "column not in file", "missing": np.nan, "distinct": np.nan})
            continue
        col = df[name]
        if pd.api.types.is_datetime64_any_dtype(col):
            observed = f"{col.min():%Y-%m-%d %H:%M} to {col.max():%Y-%m-%d %H:%M}"
        elif pd.api.types.is_numeric_dtype(col):
            observed = f"{col.min():,.4g} to {col.max():,.4g}"
        else:
            observed = f"{col.nunique()} distinct values"
        rows.append({"field": name, "type": type_text, "unit / range": unit_text, "source": source,
                     "observed range": observed, "missing": int(col.isna().sum()), "distinct": int(col.nunique())})
    return pd.DataFrame(rows)


with pd.option_context("display.max_colwidth", 60, "display.float_format", "{:,.0f}".format):
    display(attribute_table(raw_system, SYSTEM_FIELDS))

### A.7 Attribute table – process rows (specification 5.2)

In [ ]:
# Attribute table for the process rows (skipped when process.csv was not provided)
if raw_process is None:
    print("process.csv was not provided: skipping this table.")
else:
    numeric_process = [name for name, _, _, _ in PROCESS_FIELDS if name in raw_process.columns
                       and name not in ("process_name", "status")]
    raw_process[numeric_process] = raw_process[numeric_process].apply(pd.to_numeric, errors="coerce")
    print("Process rows:", len(raw_process), "| samples covered:", raw_process["sample_id"].nunique())
    with pd.option_context("display.max_colwidth", 60, "display.float_format", "{:,.0f}".format):
        display(attribute_table(raw_process, PROCESS_FIELDS))

### A.8 Class balance

In [ ]:
# Class balance of the rows that can be scored
eligible_raw = labelled_raw[labelled_raw["label"].isin(LABEL_ORDER) & ~labelled_raw["is_trimmed"] & ~labelled_raw["run_void"]]
balance = (eligible_raw.assign(target=np.where(eligible_raw["label"] == "normal", "normal (0)", "anomaly (1)"))
           .groupby(["session_id", "target"]).size().unstack(fill_value=0)
           .reindex(columns=["normal (0)", "anomaly (1)"], fill_value=0))
balance["anomaly share"] = balance["anomaly (1)"] / (balance["anomaly (1)"] + balance["normal (0)"])
display(balance)

print("Rows by stress type and intensity:")
display(eligible_raw[eligible_raw["label"].isin(STRESS_LABELS)].groupby(["label", "intensity"]).size().to_frame("rows"))

### Written part of section A

> **WRITE – how the data was collected** (machines, dates, sessions, tools, stress runs). `<your text>`

> **WRITE – class balance** (from A.8). `<your text>`

> **WRITE – limits of the data.** `<your text>`

# Section B – Statistics (rubric B)

### B.1 Descriptive statistics

In [ ]:
# Continuous numeric metric columns from the specification (identifiers and time are excluded)
METRIC_COLUMNS = [c for c in ["cpu_percent", "cpu_count", "ram_total", "ram_used", "ram_available", "ram_percent",
                              "swap_percent", "disk_used", "disk_free", "disk_percent", "disk_read_rate",
                              "disk_write_rate", "network_sent_rate", "network_recv_rate", "load_average",
                              "process_count"] if c in raw_system.columns]

percentile_points = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
stats_table = raw_system[METRIC_COLUMNS].describe(percentiles=percentile_points).T
stats_table["median"] = raw_system[METRIC_COLUMNS].median()
stats_table["variance"] = raw_system[METRIC_COLUMNS].var()
stats_table["skewness"] = raw_system[METRIC_COLUMNS].skew()
stats_table = stats_table.rename(columns={"1%": "p1", "5%": "p5", "25%": "p25", "50%": "p50",
                                          "75%": "p75", "95%": "p95", "99%": "p99"})
column_order = ["count", "mean", "median", "std", "variance", "min", "p1", "p5", "p25", "p50", "p75", "p95", "p99",
                "max", "skewness"]
display(stats_table[column_order])

### B.2 Correlation matrix

In [ ]:
# Correlation matrix of the metrics that are not constant
nonconstant = [c for c in METRIC_COLUMNS if raw_system[c].nunique(dropna=True) > 1]
corr = raw_system[nonconstant].corr()
display(corr.round(2))

# List the strongest pairs so they are easy to quote
pair_rows = []
for i, a in enumerate(nonconstant):
    for b in nonconstant[i + 1:]:
        pair_rows.append({"feature_a": a, "feature_b": b, "r": corr.loc[a, b]})
pairs = pd.DataFrame(pair_rows)
strong_pairs = pairs[pairs["r"].abs() >= 0.90].sort_values("r", key=lambda s: s.abs(), ascending=False)
print("Pairs with |r| >= 0.90:")
display(strong_pairs if len(strong_pairs) else pd.DataFrame({"note": ["none"]}))

### B.3 Medians per label

In [ ]:
# Rows that can be scored: labelled normal or stress, not trimmed, not in a void run
scored_raw = labelled_raw[labelled_raw["label"].isin(LABEL_ORDER) & ~labelled_raw["is_trimmed"] & ~labelled_raw["run_void"]]
present_labels = [l for l in LABEL_ORDER if l in set(scored_raw["label"])]
if "normal" not in present_labels:
    raise ValueError("No scorable normal rows were found. Check experiments.csv and the timestamps.")

medians = scored_raw.groupby("label")[METRIC_COLUMNS].median().T[present_labels]
print("Median of every metric per label:")
display(medians)

key_columns = [c for c in ["cpu_percent", "ram_percent", "swap_percent", "disk_write_rate", "network_recv_rate",
                           "load_average"] if c in METRIC_COLUMNS]
print("Medians by stress type and intensity:")
display(scored_raw[scored_raw["label"].isin(STRESS_LABELS)].groupby(["label", "intensity"])[key_columns].median())

# Shift of each stress type away from normal, measured in normal interquartile ranges
normal_rows = scored_raw.loc[scored_raw["label"] == "normal", METRIC_COLUMNS]
normal_iqr = (normal_rows.quantile(0.75) - normal_rows.quantile(0.25)).replace(0, np.nan)
shift = medians[[l for l in present_labels if l != "normal"]].sub(medians["normal"], axis=0).div(normal_iqr, axis=0)
print("Shift from normal (in normal IQRs):")
display(shift)

### B.4 Facts for the written inferences

In [ ]:
# Constant within each machine (a field such as cpu_count differs between machines but never changes on one)
distinct_per_host = raw_system.groupby("host_id")[METRIC_COLUMNS].nunique().max()
constant_fields = distinct_per_host[distinct_per_host <= 1].index.tolist()
print("Constant on every machine:", constant_fields)

skewed = stats_table.loc[stats_table["skewness"].abs() >= 1, "skewness"].sort_values(key=lambda s: s.abs(), ascending=False)
print("\nStrongly skewed (|skewness| >= 1):")
print(skewed.round(2).to_string() if len(skewed) else "none")

print("\nRepeating each other (|r| >= 0.90):")
if len(strong_pairs):
    for row in strong_pairs.itertuples():
        print(f"  {row.feature_a} ~ {row.feature_b}: r = {row.r:+.3f}")
else:
    print("  none")

print("\nLargest shifts from normal (in normal IQRs), top 3 per stress type:")
for label in shift.columns:
    top = shift[label].dropna().abs().sort_values(ascending=False).head(3)
    print(f"  {label:8s}", ", ".join(f"{name} ({shift.loc[name, label]:+.1f})" for name in top.index))

### Written part of section B

> **WRITE – four inferences, with numbers:** constant fields; repeated fields; skewed fields; which metric separates which stress type from normal. `<your inferences>`

# Section C – Visualisation (rubric C)

### C.1 Time series with the stress runs shaded

In [ ]:
# Pick one machine and one session: the machine with the most labelled rows and the last session
labelled_hosts = labelled_raw.loc[labelled_raw["label"].notna(), "host_id"]
plot_host = labelled_hosts.value_counts().idxmax()
plot_session = sessions.index[-1]
window_start = sessions.loc[plot_session, "start"] - pd.Timedelta(minutes=5)
window_end = sessions.loc[plot_session, "end"] + pd.Timedelta(minutes=5)
view = labelled_raw[(labelled_raw["host_id"] == plot_host) & labelled_raw["ts"].between(window_start, window_end)]
view = view.sort_values("ts")

panels = [("CPU (%)", view["cpu_percent"]), ("RAM (%)", view["ram_percent"]),
          ("log1p(disk write rate)", np.log1p(view["disk_write_rate"])),
          ("log1p(network receive rate)", np.log1p(view["network_recv_rate"]))]
session_runs = runs[runs["session_id"] == plot_session]
if "host_id" in session_runs.columns:
    session_runs = session_runs[session_runs["host_id"] == plot_host]

fig, axes = plt.subplots(len(panels), 1, figsize=(14, 9), sharex=True)
for ax, (title, series) in zip(axes, panels):
    ax.plot(view["ts"], series, linewidth=0.8, color="black")
    for r in session_runs.itertuples():
        ax.axvspan(r.started_at, r.ended_at, color=LABEL_COLORS.get(r.label, "#cccccc"), alpha=0.25, linewidth=0)
    ax.set_ylabel(title, fontsize=9)
legend_labels = LABEL_ORDER + ["cooldown"]
axes[0].legend([plt.Rectangle((0, 0), 1, 1, color=LABEL_COLORS[l], alpha=0.4) for l in legend_labels],
               legend_labels, ncol=6, loc="upper right", fontsize=8)
axes[0].set_title(f"{plot_host}, session {plot_session}: metrics with labelled runs shaded")
axes[-1].set_xlabel("time (UTC)")
plt.tight_layout()
plt.show()

> **WRITE – Inference (time series).** What reacts to what? `<your inference>`

### C.2 Histograms of the rate fields (raw and log scale)

In [ ]:
# Raw and log1p histograms for the four rate fields (negative values are impossible and ignored here)
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for j, column in enumerate(RATE_FIELDS):
    values = raw_system[column].dropna()
    values = values[values >= 0]
    axes[0, j].hist(values, bins=60, color="#277da1")
    axes[0, j].set_yscale("log")
    axes[0, j].set_title(f"{column}\nraw (bytes/s), log count axis", fontsize=9)
    axes[1, j].hist(np.log1p(values), bins=60, color="#e76f51")
    axes[1, j].set_title(f"log1p({column})", fontsize=9)
plt.tight_layout()
plt.show()

> **WRITE – Inference (histograms).** Shape of the distributions and what the log scale changes. `<your inference>`

### C.3 Scatter plot of CPU against RAM

In [ ]:
# CPU against RAM for all rows, coloured by label (background = outside every run)
points = labelled_raw.dropna(subset=["cpu_percent", "ram_percent"])
if len(points) > 20_000:
    points = points.sample(20_000, random_state=RANDOM_STATE)
points = points.assign(group=points["label"].fillna("background"))

fig, ax = plt.subplots(figsize=(8, 6))
for group in ["background"] + LABEL_ORDER + ["cooldown"]:
    part = points[points["group"] == group]
    if len(part):
        ax.scatter(part["cpu_percent"], part["ram_percent"], s=6, alpha=0.5, color=LABEL_COLORS[group],
                   label=f"{group} ({len(part)})")
ax.set_xlabel("CPU (%)")
ax.set_ylabel("RAM (%)")
ax.set_title("CPU against RAM by label")
ax.legend(markerscale=3, fontsize=8)
plt.tight_layout()
plt.show()

> **WRITE – Inference (scatter plot).** Which labels separate, which overlap with normal use? `<your inference>`

### C.4 Correlation heatmap

In [ ]:
# Heatmap of the correlation matrix computed in B.2
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, annot_kws={"size": 7}, ax=ax)
ax.set_title("Correlation between metrics")
plt.tight_layout()
plt.show()

> **WRITE – Inference (heatmap).** Which metrics move together? `<your inference>`

### C.5 Box plots per label

In [ ]:
# One box plot per metric, one box per label
box_panels = [("cpu_percent", "CPU (%)"), ("ram_percent", "RAM (%)"), ("swap_percent", "Swap (%)"),
              ("load_average", "1-minute load average"), ("log_disk_write", "log1p(disk write rate)"),
              ("log_network_recv", "log1p(network receive rate)")]
box_data = scored_raw.assign(log_disk_write=np.log1p(scored_raw["disk_write_rate"]),
                             log_network_recv=np.log1p(scored_raw["network_recv_rate"]))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (column, title) in zip(axes.ravel(), box_panels):
    groups = [box_data.loc[box_data["label"] == l, column].dropna().to_numpy() for l in present_labels]
    parts = ax.boxplot(groups, showfliers=False, patch_artist=True)
    for patch, l in zip(parts["boxes"], present_labels):
        patch.set_facecolor(LABEL_COLORS[l])
        patch.set_alpha(0.7)
    ax.set_xticks(range(1, len(present_labels) + 1))
    ax.set_xticklabels(present_labels)
    ax.set_title(title, fontsize=10)
plt.tight_layout()
plt.show()

> **WRITE – Inference (box plots).** Which metric separates which stress type? `<your inference>`

### C.6 Bar chart of the top processes

In [ ]:
# Bar charts from the process rows (skipped when process.csv was not provided)
if raw_process is None:
    print("process.csv was not provided: this chart is skipped.")
else:
    n_samples = raw_process["sample_id"].nunique()
    share = (raw_process.groupby("process_name")["sample_id"].nunique() / n_samples).sort_values(ascending=False).head(10)
    mean_cpu = raw_process.groupby("process_name")["cpu_percent"].mean().sort_values(ascending=False).head(10)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh(share.index[::-1], share.values[::-1] * 100, color="#277da1")
    axes[0].set_xlabel("% of samples in which the process is among the recorded top processes")
    axes[0].set_title("Most frequently recorded processes")
    axes[1].barh(mean_cpu.index[::-1], mean_cpu.values[::-1], color="#e76f51")
    axes[1].set_xlabel("mean CPU (% of one core) while recorded")
    axes[1].set_title("Highest mean CPU among recorded processes")
    plt.tight_layout()
    plt.show()

> **WRITE – Inference (bar chart).** Which processes dominate? `<your inference>`

# Section D – Preprocessing (rubric D)

> **Order note.** Step 3 (drop rows whose rates are missing) runs after the features are built, so the training windows match the windows the API sees.

### D.1 Steps 1 and 2 – parse, sort and drop repeated samples

In [ ]:
steps = []   # one entry per step: (step, what was done, rows before, rows after)

# Step 1: timestamps are UTC; rows with unreadable timestamps cannot be placed in time and are dropped; sort by time
d = raw_system.copy()
before = len(d)
d = d.dropna(subset=["ts"]).sort_values(["host_id", "ts"], kind="mergesort").reset_index(drop=True)
steps.append(("1", "Parse ts as UTC, drop unreadable timestamps, sort by host_id and ts", before, len(d)))

# Step 2: drop repeats of sample_id and of (host_id, ts); the API already blocks repeats, so zero is expected
before = len(d)
d = d.drop_duplicates(subset="sample_id", keep="first")
repeated_ids = before - len(d)
after_ids = len(d)
d = d.drop_duplicates(subset=["host_id", "ts"], keep="first").reset_index(drop=True)
repeated_times = after_ids - len(d)
steps.append(("2", f"Drop repeated sample_id ({repeated_ids}) and repeated (host_id, ts) ({repeated_times})", before, len(d)))

print("Step 1 rows:", before, "->", len(d) + repeated_ids + repeated_times)
print("Step 2 repeated sample_id:", repeated_ids, "| repeated (host_id, ts):", repeated_times)

### D.2 Step 4 – range checks (step 3 is only counted here)

In [ ]:
# Step 4: range checks. Each rule marks the rows that violate it.
PERCENT_FIELDS = ["cpu_percent", "ram_percent", "swap_percent", "disk_percent"]
NON_NEGATIVE_FIELDS = RATE_FIELDS + ["load_average", "process_count"]

violations = {}
bad_rows = pd.Series(False, index=d.index)
for column in PERCENT_FIELDS:
    if column in d.columns:
        broken = d[column].notna() & ((d[column] < 0) | (d[column] > 100))
        violations[column] = int(broken.sum())
        bad_rows |= broken
for column in NON_NEGATIVE_FIELDS:
    broken = d[column].notna() & (d[column] < 0)
    violations[column] = int(broken.sum())
    bad_rows |= broken
broken = d["cpu_count"].isna() | (d["cpu_count"] <= 0)
violations["cpu_count"] = int(broken.sum())
bad_rows |= broken

before = len(d)
sys_clean = d.loc[~bad_rows].reset_index(drop=True)
steps.append(("4", "Range checks: drop rows outside the allowed ranges", before, len(sys_clean)))
print("Violations per field:")
display(pd.Series(violations, name="rows").to_frame().T)

# Step 3 preview: the first sample after an agent start has no honest rate
missing_rate_rows = int(sys_clean[RATE_FIELDS].isna().any(axis=1).sum())
print(f"Step 3 preview: {missing_rate_rows} rows have at least one missing rate (they are dropped after step 9).")

### D.3 Step 5 – fields that are left out of the features

In [ ]:
# Step 5: evidence for leaving these fields out of the features
EXCLUDED_FROM_FEATURES = ["ram_total", "ram_used", "ram_available", "disk_used", "disk_free", "disk_percent"]
excluded_present = [c for c in EXCLUDED_FROM_FEATURES if c in sys_clean.columns]
evidence = pd.DataFrame({
    "distinct values per machine (max)": sys_clean.groupby("host_id")[excluded_present].nunique().max(),
    "correlation with ram_percent": sys_clean[excluded_present].corrwith(sys_clean["ram_percent"]),
})
display(evidence)
steps.append(("5", "Leave " + ", ".join(excluded_present) + " out of the features", len(sys_clean), len(sys_clean)))

### D.4 Steps 6 to 9 – the feature code (`features.py`)

Writes `features.py`, the only place where features are computed; the API imports the same file. Windows are 30 seconds, not rows.

In [ ]:
%%writefile features.py
"""Feature engineering for the Kali anomaly detector (specification 10.2).

The only place where features are computed: the notebook writes this file and the API imports it.
"""
import numpy as np
import pandas as pd

WINDOW_SECONDS = 30  # trailing window of the rolling features

# The four rate fields (bytes per second) that get a log transform
RATE_FIELDS = ["disk_read_rate", "disk_write_rate", "network_sent_rate", "network_recv_rate"]

# Final feature list, in the order the model expects it
FEATURE_NAMES = [
    "cpu_percent", "cpu_roll_mean_30s", "cpu_roll_std_30s",
    "ram_percent", "swap_delta_30s",
    "log_disk_read_rate", "log_disk_write_rate", "log_network_sent_rate", "log_network_recv_rate",
    "load_per_core", "process_count",
]

# The features that need the previous 30 seconds (used for the ablation in section F)
ROLLING_FEATURES = ["cpu_roll_mean_30s", "cpu_roll_std_30s", "swap_delta_30s"]

REQUIRED_COLUMNS = ["ts", "cpu_percent", "cpu_count", "ram_percent", "swap_percent", "load_average",
                    "process_count"] + RATE_FIELDS


def _numbers(series):
    """Column as a float array; None, NaN and unparseable values become NaN."""
    return pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)


def _window_mean_std(values, first_in_window):
    """Mean and sample std (ddof=1) of `values` over rows first_in_window[i]..i, ignoring NaN.

    Computed window by window, so a row's result depends only on its own window (pandas' incremental rolling std does not).
    """
    n = len(values)
    position = np.arange(n)
    longest = int((position - first_in_window).max()) + 1               # most rows in any window

    def value_back(back):                                               # the value `back` rows earlier, NaN outside the window
        index = position - back
        return np.where(index >= first_in_window, values[np.maximum(index, 0)], np.nan)

    with np.errstate(all="ignore"):
        count, total = np.zeros(n), np.zeros(n)
        for back in range(longest):                                     # pass 1: number of values and their sum
            v = value_back(back)
            valid = ~np.isnan(v)
            count += valid
            total += np.where(valid, v, 0.0)
        mean = np.where(count > 0, total / count, np.nan)
        squares = np.zeros(n)
        for back in range(longest):                                     # pass 2: squared deviations from the window mean
            v = value_back(back)
            squares += np.where(np.isnan(v), 0.0, (v - mean) ** 2)
        std = np.where(count > 1, np.sqrt(squares / (count - 1)), 0.0)
    return mean, std


def build_features(df):
    """Return one feature row per input row for ONE machine (raw system rows, unique index, any order)."""
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"build_features: missing columns {missing}")
    if not df.index.is_unique:
        raise ValueError("build_features: the index of df must be unique")
    if len(df) == 0:
        return pd.DataFrame(columns=FEATURE_NAMES, index=df.index, dtype=float)

    d = df.sort_values("ts", kind="mergesort")                       # time order, stable for equal timestamps
    ts = pd.DatetimeIndex(pd.to_datetime(d["ts"], utc=True))
    out = pd.DataFrame(index=d.index)

    # First row of each trailing 30-second window (t-30 s, t]. Integer clock ticks are compared,
    # so the window edge is exact whatever the datetime resolution is.
    ticks = ts.asi8
    window_ticks = int(pd.Timedelta(seconds=WINDOW_SECONDS) / pd.Timedelta(1, unit=getattr(ts, "unit", "ns")))
    first_in_window = np.searchsorted(ticks, ticks - window_ticks, side="right")

    cpu = _numbers(d["cpu_percent"])
    out["cpu_percent"] = cpu

    # Step 8: mean and standard deviation of CPU over the trailing 30 seconds (window = last 30 s up to now)
    out["cpu_roll_mean_30s"], out["cpu_roll_std_30s"] = _window_mean_std(cpu, first_in_window)

    out["ram_percent"] = _numbers(d["ram_percent"])

    # Step 9: change in swap over the trailing 30 seconds (value now minus the value at the start of the window)
    swap = _numbers(d["swap_percent"])
    out["swap_delta_30s"] = swap - swap[first_in_window]

    # Step 6: log1p of the four rate fields (rates are near zero most of the time with rare huge values)
    for field in RATE_FIELDS:
        out["log_" + field] = np.log1p(_numbers(d[field]))

    # Step 7: load per core, so that 1.0 means every core is busy
    out["load_per_core"] = _numbers(d["load_average"]) / _numbers(d["cpu_count"])
    out["process_count"] = _numbers(d["process_count"])

    return out[FEATURE_NAMES].reindex(df.index)

### D.5 Run the feature code: evidence for steps 6 to 9

In [ ]:
import importlib
import features                                   # the file written by the cell above
importlib.reload(features)                       # pick up the new version if this cell is rerun
from features import build_features, FEATURE_NAMES, ROLLING_FEATURES, WINDOW_SECONDS


def build_features_all_hosts(df):
    """Apply build_features to each machine separately and return the rows in the original order."""
    parts = [build_features(group) for _, group in df.groupby("host_id", sort=False)]
    return pd.concat(parts).reindex(df.index)


feat_all = build_features_all_hosts(sys_clean)
print("Feature rows:", feat_all.shape, "| features:", FEATURE_NAMES)
for step_no, text in [("6", "log1p of the four rate fields"), ("7", "load_per_core = load_average / cpu_count"),
                      ("8", "cpu_roll_mean_30s and cpu_roll_std_30s (trailing 30 s)"),
                      ("9", "swap_delta_30s = swap change over the trailing 30 s (replaces the swap level)")]:
    steps.append((step_no, text, len(sys_clean), len(feat_all)))

# Step 6 evidence: skewness before and after the log transform
print("\nSkewness of the rate fields:")
display(pd.DataFrame({"before log1p": sys_clean[RATE_FIELDS].skew(),
                      "after log1p": np.log1p(sys_clean[RATE_FIELDS]).skew()}))

# Steps 7 and 8 evidence: the rolling mean lags a sudden change, the raw value does not
heavy_cpu = runs[(runs["label"] == "cpu") & (runs["intensity"] == "heavy")]
if len(heavy_cpu):
    run = heavy_cpu.iloc[0]
    rows = (sys_clean["ts"] >= run.started_at - pd.Timedelta(seconds=20)) & (sys_clean["ts"] <= run.started_at + pd.Timedelta(seconds=60))
    if "host_id" in heavy_cpu.columns and pd.notna(run.get("host_id")):
        rows &= sys_clean["host_id"] == run["host_id"]
    print(f"\nCPU features around the start of the first heavy CPU run (starts {run.started_at:%H:%M:%S} UTC):")
    display(pd.concat([sys_clean.loc[rows, ["ts", "cpu_percent"]],
                       feat_all.loc[rows, ["cpu_roll_mean_30s", "cpu_roll_std_30s"]]], axis=1).head(16))

print("\nSummary of all features (before dropping rows with missing values):")
display(feat_all.describe().T)

### D.6 Step 3 – drop rows whose features are missing

In [ ]:
# Step 3 (carried out after step 9): keep only rows whose 11 features are all present
before = len(feat_all)
has_all_features = feat_all[FEATURE_NAMES].notna().all(axis=1)
steps.append(("3", "Drop rows with a missing rate or feature (carried out after step 9)", before, int(has_all_features.sum())))
print(f"Rows before: {before} | rows with a missing feature: {int((~has_all_features).sum())} | rows kept: {int(has_all_features.sum())}")

### D.7 Step 10 – attach the labels and mark usable rows

`score_eligible` rows are scored; `train_eligible` rows (background and normal, never stress or cooldown) are used for fitting.

In [ ]:
# Step 10: attach labels to the cleaned rows, then keep only the rows with complete features
labelled_clean, run_summary = attach_labels(sys_clean, runs)
meta_columns = ["sample_id", "host_id", "ts", "run_id", "session_id", "label", "intensity", "is_trimmed",
                "run_void", "in_session_gap"]
model_df = pd.concat([labelled_clean[meta_columns], feat_all[FEATURE_NAMES]], axis=1).loc[has_all_features]
model_df = model_df.sort_values("ts", kind="mergesort").reset_index(drop=True)

# Target: 1 = anomaly (stress), 0 = normal, NaN = never scored (background, cooldown, unknown labels)
model_df["y"] = np.select([model_df["label"].isin(STRESS_LABELS), model_df["label"].eq("normal")], [1.0, 0.0], default=np.nan)
model_df["score_eligible"] = model_df["y"].notna() & ~model_df["is_trimmed"] & ~model_df["run_void"]
model_df["train_eligible"] = ((model_df["label"].isna() | model_df["label"].eq("normal"))
                              & ~model_df["run_void"] & ~model_df["in_session_gap"])

# What happens to each row
category = np.select(
    [model_df["run_void"], model_df["in_session_gap"], model_df["label"].eq("cooldown"),
     model_df["label"].notna() & ~model_df["label"].isin(LABEL_ORDER + ["cooldown"]),
     model_df["label"].isin(LABEL_ORDER) & model_df["is_trimmed"], model_df["score_eligible"]],
    ["in a void run (excluded)", "unlabelled inside a session (excluded)", "cooldown (never scored)",
     "unknown label (excluded)", "trimmed run edge (never scored)", "scored"],
    default="unlabelled background")
print("What happens to the", len(model_df), "rows with complete features:")
display(pd.Series(category).value_counts().to_frame("rows"))
print("Scored rows by target:", model_df.loc[model_df["score_eligible"], "y"].map({0.0: "normal (0)", 1.0: "anomaly (1)"}).value_counts().to_dict())
print("Rows usable for training (background + normal):", int(model_df["train_eligible"].sum()))

steps.append(("10", "Attach labels; remove cooldown, trimmed and void rows from the scored set (rows = scored rows)",
              len(model_df), int(model_df["score_eligible"].sum())))
assert not model_df[FEATURE_NAMES].isna().any().any(), "model_df must not contain missing feature values"

### D.8 Step 11 – scaling inside the LOF pipeline only

In [ ]:
# One function builds every model in this notebook, so tuning, evaluation and deployment cannot disagree
def resolve_max_samples(max_samples, n_rows):
    """Isolation Forest cannot draw a subsample larger than the training set."""
    return min(int(max_samples), n_rows) if isinstance(max_samples, (int, np.integer)) else max_samples


def make_pipeline(family, params, contamination, n_train=None):
    """Return an UNFITTED scikit-learn pipeline for one configuration (n_train caps size-dependent settings)."""
    if family == "iforest":
        max_samples = resolve_max_samples(params["max_samples"], n_train) if n_train else params["max_samples"]
        model = IsolationForest(n_estimators=params["n_estimators"], max_samples=max_samples,
                                max_features=params["max_features"], contamination=contamination,
                                random_state=RANDOM_STATE)
        return Pipeline([("model", model)])
    if family == "lof":
        neighbours = min(params["n_neighbors"], n_train - 1) if n_train else params["n_neighbors"]
        model = LocalOutlierFactor(n_neighbors=neighbours, metric=params["metric"], novelty=True,
                                   contamination=contamination)
        return Pipeline([("scaler", StandardScaler()), ("model", model)])
    raise ValueError(f"unknown model family: {family}")


steps.append(("11", "StandardScaler inside the LOF pipeline only (fitted on training rows when the pipeline is fitted)", len(model_df), len(model_df)))
print(make_pipeline("iforest", {"n_estimators": 100, "max_samples": "auto", "max_features": 1.0}, "auto"))
print(make_pipeline("lof", {"n_neighbors": 20, "metric": "euclidean"}, "auto"))

### D.9 Row counts for every step

In [ ]:
# One table with the row counts before and after every step, in step order
steps_table = pd.DataFrame(steps, columns=["step", "what was done", "rows before", "rows after"])
steps_table["rows removed"] = steps_table["rows before"] - steps_table["rows after"]
steps_table = steps_table.sort_values("step", key=lambda s: s.astype(int)).reset_index(drop=True)
with pd.option_context("display.max_colwidth", 110, "display.float_format", "{:,.0f}".format):
    display(steps_table)

### Written part of section D

> **WRITE – the reason for each step**, in your own words and with a number where you can.

| Step | Operation | Reason |
|---|---|---|
| 1 | Parse `ts` as UTC, sort by machine and time | `<reason>` |
| 2 | Drop repeated `sample_id` and (`host_id`, `ts`) | `<reason>` |
| 3 | Drop rows whose rates are missing | `<reason>` |
| 4 | Range checks | `<reason>` |
| 5 | Leave out RAM total, used, available and the disk-space fields | `<reason>` |
| 6 | `log1p` of the four rate fields | `<reason>` |
| 7 | `load_per_core` | `<reason>` |
| 8 | 30-second rolling mean and standard deviation of CPU | `<reason>` |
| 9 | Swap change over 30 seconds instead of the swap level | `<reason>` |
| 10 | Attach labels; remove cooldown and trimmed rows from scored sets | `<reason>` |
| 11 | `StandardScaler` inside the LOF pipeline only | `<reason>` |

# Section E – Split and cross-validation (rubric E)

> Rows are never shuffled. The last labelled session is the test set (opened once, in section G). Every earlier session is one fold, fitted on rows recorded before it.

### E.1 Sessions and their roles

In [ ]:
# E.1 give every session a role: the last one is the test session, every earlier one is a cross-validation fold
session_ids = list(sessions.index)                    # time order (sessions was sorted by start time in section A)
if len(session_ids) < 2:
    raise ValueError("At least two labelled sessions are needed: one or more for cross-validation and one test session.")
TEST_SESSION = session_ids[-1]
FOLD_SESSIONS = session_ids[:-1]
if len(FOLD_SESSIONS) < 3:
    print(f"NOTE: only {len(FOLD_SESSIONS)} session(s) before the test session, so cross-validation has {len(FOLD_SESSIONS)} fold(s).")

scored = model_df[model_df["score_eligible"]]
per_session = scored.groupby("session_id")["y"].agg(rows="size", anomaly="sum")
session_table = sessions.copy()
session_table["role"] = ["TEST (opened once, section G)" if s == TEST_SESSION else f"fold {i + 1}" for i, s in enumerate(session_ids)]
session_table["scored anomaly"] = per_session["anomaly"].reindex(session_table.index).fillna(0).astype(int)
session_table["scored normal"] = (per_session["rows"] - per_session["anomaly"]).reindex(session_table.index).fillna(0).astype(int)
session_table["anomaly share"] = session_table["scored anomaly"] / (session_table["scored anomaly"] + session_table["scored normal"])
session_table["void runs"] = run_summary[run_summary["void"]].groupby("session_id").size().reindex(session_table.index).fillna(0).astype(int)
display(session_table[["order", "start", "end", "role", "scored normal", "scored anomaly", "anomaly share", "void runs"]])

### E.2 The folds and the number of rows in each

In [ ]:
# E.2 build the expanding-window folds and the test split; both are plain lists of row positions
MIN_TRAIN_ROWS = 300                                  # a fold with fewer training rows is skipped


def make_expanding_folds(frame, session_table, fold_sessions):
    """One fold per session: fit on training rows recorded before it begins, score its labelled rows."""
    folds = []
    for session_id in fold_sessions:
        start = session_table.loc[session_id, "start"]
        train_idx = np.flatnonzero((frame["train_eligible"] & (frame["ts"] < start)).to_numpy())
        score_idx = np.flatnonzero((frame["score_eligible"] & (frame["session_id"] == session_id)).to_numpy())
        folds.append({"session": session_id, "start": start, "end": session_table.loc[session_id, "end"],
                      "train_idx": train_idx, "score_idx": score_idx})
    return folds


def describe_fold(name, fold):
    """One row of the fold table."""
    train, scored_rows = model_df.iloc[fold["train_idx"]], model_df.iloc[fold["score_idx"]]
    return {"fold": name, "scored session": fold["session"], "training rows": len(train),
            "of which background": int(train["label"].isna().sum()), "of which labelled normal": int((train["label"] == "normal").sum()),
            "training from": train["ts"].min(), "training until": train["ts"].max(),
            "scored rows": len(scored_rows), "scored normal": int((scored_rows["y"] == 0).sum()),
            "scored anomaly": int((scored_rows["y"] == 1).sum())}


def usable(fold):
    """A fold needs enough training rows and both classes among the scored rows."""
    return len(fold["train_idx"]) >= MIN_TRAIN_ROWS and model_df["y"].iloc[fold["score_idx"]].nunique() == 2


all_folds = make_expanding_folds(model_df, session_table, FOLD_SESSIONS)
folds = [f for f in all_folds if usable(f)]
skipped = [f["session"] for f in all_folds if not usable(f)]
if skipped:
    print("WARNING: skipped folds (too few training rows or only one class to score):", skipped)
if not folds:
    raise ValueError("No usable cross-validation fold. Check the sessions, the labels and the amount of background data.")

test_fold = make_expanding_folds(model_df, session_table, [TEST_SESSION])[0]
if not usable(test_fold):
    raise ValueError("The test session has too few training rows before it, or lacks normal or anomaly rows to score.")

fold_table = pd.DataFrame([describe_fold(f"fold {i + 1}", f) for i, f in enumerate(folds)] + [describe_fold("TEST", test_fold)])
with pd.option_context("display.max_colwidth", 40, "display.float_format", "{:,.0f}".format):
    display(fold_table)

### E.3 Checks that nothing leaks

In [ ]:
# E.3 assertions: no leakage between fitting and scoring, and the test session stays untouched
def check_no_leakage(frame, fold_list, test):
    """Raise an AssertionError that names the problem if any rule of the split is broken."""
    test_start = test["start"]
    for fold in fold_list + [test]:
        train, scored_rows, name = frame.iloc[fold["train_idx"]], frame.iloc[fold["score_idx"]], fold["session"]
        assert np.intersect1d(fold["train_idx"], fold["score_idx"]).size == 0, f"{name}: a row is used for fitting and scoring"
        assert (train["ts"] < fold["start"]).all(), f"{name}: a training row was recorded after the session began"
        assert not train["label"].isin(STRESS_LABELS + ["cooldown"]).any(), f"{name}: a stress or cooldown row is a training row"
        assert not train["run_void"].any() and not train["in_session_gap"].any(), f"{name}: an excluded row is a training row"
        assert (scored_rows["session_id"] == name).all() and scored_rows["score_eligible"].all(), f"{name}: a scored row is not scorable"
        assert not scored_rows["is_trimmed"].any(), f"{name}: a trimmed row is scored"
        assert set(train["run_id"].dropna()).isdisjoint(set(scored_rows["run_id"].dropna())), f"{name}: a run is split between fitting and scoring"
    for fold in fold_list:
        assert np.intersect1d(test["score_idx"], fold["train_idx"]).size == 0, "the test session is a training row of a fold"
        assert np.intersect1d(test["score_idx"], fold["score_idx"]).size == 0, "the test session is scored in a fold"
        assert frame["ts"].iloc[fold["train_idx"]].max() < test_start, "a fold trains on rows recorded after the test session began"
    return True


print("All leakage checks passed:", check_no_leakage(model_df, folds, test_fold))

### E.4 Timeline of the folds

In [ ]:
# E.4 draw the training span and the scored session of every fold on one time axis
def naive_utc(timestamp):
    """Timestamp without a time zone (UTC), so that matplotlib places it on a date axis."""
    return pd.Timestamp(timestamp).tz_convert("UTC").tz_localize(None)


plot_folds = folds + [test_fold]
fig, ax = plt.subplots(figsize=(14, 0.8 * len(plot_folds) + 2))
for row, fold in enumerate(plot_folds):
    y_pos = len(plot_folds) - 1 - row
    train_ts = model_df["ts"].iloc[fold["train_idx"]]
    colour = "#c1121f" if fold is test_fold else "#f4a261"
    ax.hlines(y_pos, naive_utc(train_ts.min()), naive_utc(train_ts.max()), color="#2a9d8f", linewidth=12, alpha=0.75)
    ax.hlines(y_pos, naive_utc(fold["start"]), naive_utc(fold["end"]), color=colour, linewidth=12)
    ax.text(naive_utc(fold["start"]), y_pos + 0.25, fold["session"], fontsize=8, ha="left")
ax.set_yticks(range(len(plot_folds)))
ax.set_yticklabels([("TEST: " if f is test_fold else f"fold {i + 1}: ") + f["session"] for i, f in enumerate(plot_folds)][::-1])
ax.set_ylim(-0.6, len(plot_folds) - 0.2)
ax.set_title("Expanding-window cross-validation over sessions (UTC)", pad=28)
ax.legend([plt.Line2D([0], [0], color=c, linewidth=8) for c in ("#2a9d8f", "#f4a261", "#c1121f")],
          ["training rows", "scored session", "test session"], loc="lower left", bbox_to_anchor=(0, 1.0), ncol=3, fontsize=8)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Written part of section E

> **WRITE – remarks on cross-validation:** why shuffled k-fold is wrong; why the window expands forward; why whole sessions are the unit; how an unsupervised model is scored; limits (use the fold table). `<your remarks>`

# Section F – Training and tuning (rubric F)

> Selection: best mean F1; settings within 0.01 are tied, then higher average precision, then the simpler setting. `contamination` only moves the threshold, so each setting is fitted once per fold (checked in F.3).

### F.1 The search space

In [ ]:
# F.1 the grids of specification 10.4; QUICK_RUN shrinks them for a trial run
IF_PARAMS = ["n_estimators", "max_samples", "max_features"]       # fitted parameters of Isolation Forest
LOF_PARAMS = ["n_neighbors", "metric"]                            # fitted parameters of LOF
IF_COLS, LOF_COLS = IF_PARAMS + ["contamination"], LOF_PARAMS + ["contamination"]

IF_BASELINE = {"n_estimators": 100, "max_samples": "auto", "max_features": 1.0}      # contamination = "auto"
LOF_BASELINE = {"n_neighbors": 20, "metric": "euclidean"}                           # contamination = "auto"
RULE_BASELINE_PERCENTILE = 99

if QUICK_RUN:
    IF_GRID = {"n_estimators": [100], "max_samples": [256, 512], "max_features": [0.75, 1.0]}
    LOF_GRID = {"n_neighbors": [20, 35], "metric": ["euclidean"]}
    CONTAMINATIONS = [0.01, 0.05]
    RULE_PERCENTILES = [95, 99]
else:
    IF_GRID = {"n_estimators": [100, 200, 400], "max_samples": [256, 512, 1024], "max_features": [0.5, 0.75, 1.0]}
    LOF_GRID = {"n_neighbors": [10, 20, 35, 50], "metric": ["euclidean", "manhattan"]}
    CONTAMINATIONS = [0.005, 0.01, 0.02, 0.05]
    RULE_PERCENTILES = [95, 99, 99.5]


def expand_grid(grid):
    """All combinations of a grid given as {name: [values]}."""
    return [dict(zip(grid, values)) for values in product(*grid.values())]


n_if, n_lof = len(expand_grid(IF_GRID)), len(expand_grid(LOF_GRID))
print("QUICK_RUN =", QUICK_RUN)
print(f"Isolation Forest: {n_if} fitted settings x {len(CONTAMINATIONS)} contamination values = {n_if * len(CONTAMINATIONS)} configurations")
print(f"LOF             : {n_lof} fitted settings x {len(CONTAMINATIONS)} contamination values = {n_lof * len(CONTAMINATIONS)} configurations")
print(f"Percentile rule : {len(RULE_PERCENTILES)} percentiles | folds: {len(folds)} | fits per fold: {n_if + n_lof + 2}")

### F.2 Helper functions

In [ ]:
# F.2 helpers shared by tuning, evaluation and deployment
EXCEEDANCE_EPS = 1e-6                                 # the "small constant" of the exceedance formula (specification 11.2)
X_ALL = model_df[FEATURE_NAMES].reset_index(drop=True)    # features of every row of model_df, in the model's order
Y_ALL = model_df["y"].to_numpy()                      # 1 = anomaly, 0 = normal, NaN = never scored
LABEL_ALL = model_df["label"].astype(object).to_numpy()
INTENSITY_ALL = model_df["intensity"].astype(object).to_numpy()


def subsample_rows(positions, limit, seed=RANDOM_STATE):
    """Random subsample (fixed seed) of row positions when there are more than `limit` of them."""
    if limit is None or len(positions) <= limit:
        return positions
    return np.sort(np.random.RandomState(seed).choice(positions, size=limit, replace=False))


def fit_pipeline(family, params, contamination, X_train):
    """Fit one scikit-learn pipeline; returns the pipeline and the seconds the fit took."""
    started = time.perf_counter()
    pipe = make_pipeline(family, params, contamination, n_train=len(X_train)).fit(X_train)
    return pipe, time.perf_counter() - started


def reference_scores(pipe, family, X_train):
    """Anomaly scores of the training rows that scikit-learn itself uses to place the contamination threshold."""
    if family == "lof":
        return -pipe.named_steps["model"].negative_outlier_factor_     # LOF compares each training row with the others
    return -pipe.score_samples(X_train)


def threshold_for(contamination, reference, pipe):
    """Anomaly score above which a row is flagged: sklearn's fixed offset for 'auto', else a training percentile."""
    if contamination == "auto":
        return -pipe.named_steps["model"].offset_
    return -np.percentile(-reference, 100.0 * contamination)           # mirrors np.percentile(score_samples, 100 * c)


class PercentileRule:
    """Non-ML baseline: flag a row when ANY feature is above its P-th percentile in the training rows."""

    def __init__(self, percentile):
        self.percentile = percentile

    def fit(self, X_train):
        values = np.asarray(X_train, dtype=float)
        self.p50_ = np.percentile(values, 50, axis=0)
        self.p99_ = np.percentile(values, 99, axis=0)
        self.limit_ = np.percentile(values, self.percentile, axis=0)
        return self

    def exceedance(self, X):
        """(value - p50) / max(p99 - p50, small constant), as in specification 11.2."""
        return (np.asarray(X, dtype=float) - self.p50_) / np.maximum(self.p99_ - self.p50_, EXCEEDANCE_EPS)

    def score(self, X):
        """Largest exceedance over the features (higher = more unusual)."""
        return self.exceedance(X).max(axis=1)

    def flag(self, X):
        """True when any feature is above its limit."""
        return (np.asarray(X, dtype=float) > self.limit_).any(axis=1)


def score_metrics(y_true, flagged, score, labels=None):
    """Every metric of specification 10.5 that can be computed for one scored set."""
    y, flagged = np.asarray(y_true).astype(int), np.asarray(flagged).astype(int)
    both_classes = len(np.unique(y)) == 2
    out = {"accuracy": accuracy_score(y, flagged),
           "balanced_accuracy": balanced_accuracy_score(y, flagged) if both_classes else np.nan,
           "precision": precision_score(y, flagged, zero_division=0), "recall": recall_score(y, flagged, zero_division=0),
           "f1": f1_score(y, flagged, zero_division=0),
           "roc_auc": roc_auc_score(y, score) if both_classes else np.nan,
           "average_precision": average_precision_score(y, score) if y.any() else np.nan,
           "fpr": flagged[y == 0].mean() if (y == 0).any() else np.nan}
    if labels is not None:
        for kind in STRESS_LABELS:
            in_kind = np.asarray(labels) == kind
            out[f"recall_{kind}"] = flagged[in_kind].mean() if in_kind.any() else np.nan
    return out


def describe_setting(params, contamination):
    """Readable key of one configuration, used to group the results."""
    return ", ".join(f"{key}={value}" for key, value in {**params, "contamination": contamination}.items())


def cross_validate_model(family, grid, contaminations, fold_list, columns=None, verbose=True):
    """Expanding-window CV for 'iforest' or 'lof': one fit per setting and fold, every contamination as a threshold."""
    columns = list(columns or FEATURE_NAMES)
    cap = LOF_MAX_TRAIN_ROWS if family == "lof" else IF_MAX_TRAIN_ROWS
    settings, rows, started = expand_grid(grid), [], time.perf_counter()
    for number, params in enumerate(settings, start=1):
        for fold in fold_list:
            train_pos = subsample_rows(fold["train_idx"], cap)
            X_train, X_score = X_ALL.iloc[train_pos][columns], X_ALL.iloc[fold["score_idx"]][columns]
            pipe, fit_seconds = fit_pipeline(family, params, "auto", X_train)
            reference = reference_scores(pipe, family, X_train)
            score = -pipe.score_samples(X_score)
            for c in contaminations:
                flagged = score > threshold_for(c, reference, pipe)
                rows.append({"family": family, **params, "contamination": c, "setting": describe_setting(params, c),
                             "fold": fold["session"], "n_train": len(train_pos), "fit_seconds": fit_seconds,
                             **score_metrics(Y_ALL[fold["score_idx"]], flagged, score, LABEL_ALL[fold["score_idx"]])})
        if verbose and (number % max(1, len(settings) // 5) == 0 or number == len(settings)):
            print(f"  {family}: {number}/{len(settings)} settings fitted ({time.perf_counter() - started:.0f} s)")
    return pd.DataFrame(rows)


def cross_validate_rule(percentiles, fold_list, columns=None):
    """The same folds for the percentile rule."""
    columns, rows = list(columns or FEATURE_NAMES), []
    for p in percentiles:
        for fold in fold_list:
            X_train, X_score = X_ALL.iloc[fold["train_idx"]][columns], X_ALL.iloc[fold["score_idx"]][columns]
            rule = PercentileRule(p).fit(X_train)
            rows.append({"family": "rule", "percentile": p, "setting": f"percentile={p}", "fold": fold["session"],
                         "n_train": len(X_train), "fit_seconds": 0.0,
                         **score_metrics(Y_ALL[fold["score_idx"]], rule.flag(X_score), rule.score(X_score), LABEL_ALL[fold["score_idx"]])})
    return pd.DataFrame(rows)


SUMMARY_METRICS = ["f1", "average_precision", "precision", "recall", "roc_auc", "accuracy", "balanced_accuracy", "fpr"] + [f"recall_{k}" for k in STRESS_LABELS]


def summarise_cv(frame, param_columns):
    """Mean over the folds (and the fold-to-fold standard deviation of F1 and average precision), one row per setting."""
    grouped = frame.groupby("setting", sort=False)
    out = grouped[SUMMARY_METRICS].mean().add_suffix("_mean").rename(columns={"average_precision_mean": "ap_mean"})
    out["f1_std"], out["ap_std"], out["folds"] = grouped["f1"].std(), grouped["average_precision"].std(), grouped.size()
    return pd.concat([grouped[param_columns].first(), out], axis=1).reset_index()


def select_setting(summary, complexity, tolerance=0.01):
    """Specification 10.4: best mean F1; ties (within 0.01) go to the higher average precision, then the simpler setting."""
    best_f1 = summary["f1_mean"].max()
    tied_f1 = summary[summary["f1_mean"] >= best_f1 - tolerance]
    tied_ap = tied_f1[tied_f1["ap_mean"] >= tied_f1["ap_mean"].max() - tolerance]
    chosen = min(tied_ap.to_dict("records"), key=lambda r: (complexity(r), -r["f1_mean"]))
    return chosen, {"best_f1": best_f1, "tied_on_f1": len(tied_f1), "tied_after_ap": len(tied_ap)}


def complexity_iforest(r):
    """Smaller = simpler: fewer trees, then smaller subsample, then fewer features per tree."""
    return (r["n_estimators"], 256 if r["max_samples"] == "auto" else r["max_samples"], r["max_features"])


def complexity_lof(r):
    """Smaller = simpler: Euclidean before Manhattan, then fewer neighbours."""
    return (0 if r["metric"] == "euclidean" else 1, r["n_neighbors"])


def complexity_rule(r):
    """The percentile rule has no complexity ordering; ties go to the higher F1."""
    return (0,)


print("Helpers defined. Exceedance constant:", EXCEEDANCE_EPS, "| LOF row cap:", LOF_MAX_TRAIN_ROWS, "| Isolation Forest row cap:", IF_MAX_TRAIN_ROWS)

### F.3 Check: "fit once, apply many thresholds" equals scikit-learn

In [ ]:
# F.3 prove the speed-up is exact: compare with sklearn's own predict
check_fold = folds[-1]
for family, params in [("iforest", {"n_estimators": 100, "max_samples": 256, "max_features": 1.0}),
                       ("lof", {"n_neighbors": 20, "metric": "euclidean"})]:
    train_pos = subsample_rows(check_fold["train_idx"], LOF_MAX_TRAIN_ROWS if family == "lof" else IF_MAX_TRAIN_ROWS)
    X_train, X_score = X_ALL.iloc[train_pos], X_ALL.iloc[check_fold["score_idx"]]
    auto_pipe, _ = fit_pipeline(family, params, "auto", X_train)
    reference = reference_scores(auto_pipe, family, X_train)
    score = -auto_pipe.score_samples(X_score)

    for c in (0.005, 0.02, 0.05):
        direct_pipe, _ = fit_pipeline(family, params, c, X_train)                  # sklearn is told the contamination
        direct_flags = direct_pipe.predict(X_score) == -1
        shortcut_flags = score > threshold_for(c, reference, auto_pipe)            # percentile threshold on the same scores
        assert (direct_flags == shortcut_flags).all(), f"{family}: shortcut differs from sklearn at contamination {c}"
        assert np.allclose(-direct_pipe.score_samples(X_score), score), f"{family}: scores differ between fits"
    auto_flags = auto_pipe.predict(X_score) == -1                                    # the baseline uses sklearn's fixed offset
    assert (auto_flags == (score > threshold_for("auto", reference, auto_pipe))).all(), f"{family}: 'auto' threshold differs"
    print(f"{family:8s} shortcut identical to scikit-learn at contamination 0.005, 0.02, 0.05 and 'auto' "
          f"({len(X_train)} training rows, {len(X_score)} scored rows)")

### F.4 Cross-validate the baselines and the full grids

In [ ]:
# F.4 run the cross-validation for all models: baselines first, then the grids (all on the same folds)
print("Folds:", [f["session"] for f in folds], "| QUICK_RUN =", QUICK_RUN)
started = time.perf_counter()
cv_if_base = cross_validate_model("iforest", {k: [v] for k, v in IF_BASELINE.items()}, ["auto"], folds, verbose=False)
cv_lof_base = cross_validate_model("lof", {k: [v] for k, v in LOF_BASELINE.items()}, ["auto"], folds, verbose=False)
cv_rule_base = cross_validate_rule([RULE_BASELINE_PERCENTILE], folds)
cv_if = cross_validate_model("iforest", IF_GRID, CONTAMINATIONS, folds)
cv_lof = cross_validate_model("lof", LOF_GRID, CONTAMINATIONS, folds)
cv_rule = cross_validate_rule(RULE_PERCENTILES, folds)
print(f"Cross-validation finished in {time.perf_counter() - started:.0f} s.")
print("Result rows (one per setting and fold):", {"IF": len(cv_if), "LOF": len(cv_lof), "rule": len(cv_rule)})

### F.5 Selection

In [ ]:
# F.5 apply the selection rule of specification 10.4 to every model
sum_if, sum_lof, sum_rule = summarise_cv(cv_if, IF_COLS), summarise_cv(cv_lof, LOF_COLS), summarise_cv(cv_rule, ["percentile"])
best_if, info_if = select_setting(sum_if, complexity_iforest)
best_lof, info_lof = select_setting(sum_lof, complexity_lof)
best_rule, info_rule = select_setting(sum_rule, complexity_rule)

show_columns = ["f1_mean", "f1_std", "ap_mean", "precision_mean", "recall_mean", "fpr_mean"]
for title, summary, cols, best, info in [("Isolation Forest", sum_if, IF_COLS, best_if, info_if),
                                         ("LOF", sum_lof, LOF_COLS, best_lof, info_lof),
                                         ("Percentile rule", sum_rule, ["percentile"], best_rule, info_rule)]:
    print(f"\n{title}: best mean F1 {info['best_f1']:.3f} | {info['tied_on_f1']} setting(s) within 0.01 | "
          f"{info['tied_after_ap']} left after the average-precision step")
    print("  CHOSEN ->", best["setting"], f"(mean F1 {best['f1_mean']:.3f}, average precision {best['ap_mean']:.3f})")
    display(summary.sort_values("f1_mean", ascending=False).head(5)[cols + show_columns])

IF_BEST_PARAMS = {k: best_if[k] for k in IF_PARAMS}
LOF_BEST_PARAMS = {k: best_lof[k] for k in LOF_PARAMS}

# warn when a chosen value sits at the edge of its grid: a better value might lie outside the specified range
for title, record, grid in [("Isolation Forest", best_if, IF_GRID), ("LOF", best_lof, LOF_GRID)]:
    for key, values in {**grid, "contamination": CONTAMINATIONS}.items():
        numbers = [v for v in values if isinstance(v, (int, float))]
        if len(numbers) > 1 and record[key] in (min(numbers), max(numbers)):
            print(f"NOTE: {title}: {key} = {record[key]} is at the edge of its grid {values}; say what that implies in the written part.")

### F.6 Before and after tuning

The tuned setting is the best of many on these folds; the test session in section G is the fair check.

In [ ]:
# F.6 before-and-after table and the paired fold-by-fold gain
def summary_line(label, record):
    """One row of the before-and-after table."""
    return {"model": label, "setting": record["setting"], "F1 mean": record["f1_mean"], "F1 SD": record["f1_std"],
            "avg precision": record["ap_mean"], "precision": record["precision_mean"], "recall": record["recall_mean"],
            "ROC-AUC": record["roc_auc_mean"], "false-positive rate": record["fpr_mean"]}


def first_record(frame, cols):
    """The single baseline setting as a summary record."""
    return summarise_cv(frame, cols).iloc[0].to_dict()


def paired_gain(baseline_frame, tuned_frame, setting):
    """Tuned F1 minus baseline F1, fold by fold."""
    baseline = baseline_frame.set_index("fold")["f1"]
    tuned = tuned_frame[tuned_frame["setting"] == setting].set_index("fold")["f1"]
    return tuned - baseline.reindex(tuned.index)


def verdict(mean_gain, spread):
    """Compare the mean gain with the spread of the fold-by-fold gains (a rule of thumb, not a significance test)."""
    if np.isfinite(spread) and spread == 0 and mean_gain == 0:
        return "no change (same setting as the baseline)"
    if not np.isfinite(spread) or spread == 0:
        return "too few folds to judge"
    ratio = mean_gain / spread
    if ratio >= 2:
        return "clearly larger than the fold-to-fold variation"
    if ratio >= 1:
        return "about as large as the fold-to-fold variation"
    return "within the fold-to-fold variation" if ratio > -1 else "worse than the baseline"


before_after = pd.DataFrame([
    summary_line("Isolation Forest - default", first_record(cv_if_base, IF_COLS)), summary_line("Isolation Forest - tuned", best_if),
    summary_line("LOF - default", first_record(cv_lof_base, LOF_COLS)), summary_line("LOF - tuned", best_lof),
    summary_line("Percentile rule - 99th (default)", first_record(cv_rule_base, ["percentile"])), summary_line("Percentile rule - selected", best_rule)])
with pd.option_context("display.max_colwidth", 95):
    display(before_after)

gain_rows = []
for name, base_frame, tuned_frame, record in [("Isolation Forest", cv_if_base, cv_if, best_if), ("LOF", cv_lof_base, cv_lof, best_lof),
                                              ("Percentile rule", cv_rule_base, cv_rule, best_rule)]:
    gains = paired_gain(base_frame, tuned_frame, record["setting"])
    gain_rows.append({"model": name, "mean F1 gain": gains.mean(), "SD of the fold gains": gains.std(),
                      "verdict": verdict(gains.mean(), gains.std())})
display(pd.DataFrame(gain_rows))

### F.7 Validation score against each hyperparameter

In [ ]:
# F.7 validation curves for the tuned parameters (figure saved for the report)
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)


def plot_validation_curve(ax, summary, best, vary, param_cols, title):
    """Mean CV F1 and average precision against one parameter, the other parameters held at their selected values."""
    keep = np.ones(len(summary), dtype=bool)
    for column in param_cols:
        if column != vary:
            keep &= (summary[column] == best[column]).to_numpy()
    part = summary[keep].sort_values(vary)
    positions = np.arange(len(part))
    ax.errorbar(positions, part["f1_mean"], yerr=part["f1_std"].fillna(0), marker="o", capsize=3, label="F1 (mean, SD)")
    ax.plot(positions, part["ap_mean"], marker="s", linestyle="--", color="grey", label="average precision")
    selected = np.flatnonzero((part[vary] == best[vary]).to_numpy())
    ax.scatter(selected, part["f1_mean"].to_numpy()[selected], s=260, facecolors="none", edgecolors="red", linewidths=2, zorder=5)
    ax.set_xticks(positions)
    ax.set_xticklabels([str(v) for v in part[vary]])
    ax.set_xlabel(vary)
    ax.set_ylabel("validation score")
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, 1.05)


fig, axes = plt.subplots(2, 3, figsize=(16, 8))
plot_validation_curve(axes[0, 0], sum_if, best_if, "contamination", IF_COLS, "Isolation Forest: contamination")
plot_validation_curve(axes[0, 1], sum_if, best_if, "max_samples", IF_COLS, "Isolation Forest: max_samples")
plot_validation_curve(axes[0, 2], sum_if, best_if, "n_estimators", IF_COLS, "Isolation Forest: n_estimators")
plot_validation_curve(axes[1, 0], sum_if, best_if, "max_features", IF_COLS, "Isolation Forest: max_features")
plot_validation_curve(axes[1, 1], sum_lof, best_lof, "n_neighbors", LOF_COLS, "LOF: n_neighbors")
plot_validation_curve(axes[1, 2], sum_lof, best_lof, "contamination", LOF_COLS, "LOF: contamination")
axes[0, 0].legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "validation_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### F.8 Ablation: with and without the rolling features

In [ ]:
# F.8 ablation: the same selected setting, with and without the three rolling features
NO_ROLLING = [c for c in FEATURE_NAMES if c not in ROLLING_FEATURES]
ablation_rows = []
for model_name, family, record, param_cols in [("Isolation Forest (tuned)", "iforest", best_if, IF_PARAMS), ("LOF (tuned)", "lof", best_lof, LOF_PARAMS)]:
    for variant, cols in [("all 11 features", FEATURE_NAMES), (f"without rolling features ({len(NO_ROLLING)} left)", NO_ROLLING)]:
        frame = cross_validate_model(family, {k: [record[k]] for k in param_cols}, [record["contamination"]], folds, columns=cols, verbose=False)
        line = summarise_cv(frame, param_cols + ["contamination"]).iloc[0]
        ablation_rows.append({"model": model_name, "features": variant, "F1 mean": line["f1_mean"], "F1 SD": line["f1_std"],
                              "avg precision": line["ap_mean"], "recall": line["recall_mean"], "false-positive rate": line["fpr_mean"],
                              **{f"recall {k}": line[f"recall_{k}_mean"] for k in STRESS_LABELS}})
ablation = pd.DataFrame(ablation_rows)
display(ablation)

### F.9 The selected settings, fold by fold

In [ ]:
# F.9 per-fold results of the selected settings (this is the spread that the written part refers to)
fold_view = []
for name, frame, record in [("Isolation Forest (tuned)", cv_if, best_if), ("LOF (tuned)", cv_lof, best_lof), ("Percentile rule", cv_rule, best_rule)]:
    part = frame[frame["setting"] == record["setting"]]
    fold_view.append(part[["fold", "n_train", "precision", "recall", "f1", "average_precision", "roc_auc", "fpr"]].assign(model=name))
display(pd.concat(fold_view)[["model", "fold", "n_train", "precision", "recall", "f1", "average_precision", "roc_auc", "fpr"]].reset_index(drop=True))

### Written part of section F

> **WRITE – why each chosen value makes sense** (use F.5 and the curves of F.7; say what it implies if a value sits at the edge of its grid). `<your justification>`

> **WRITE – did tuning help beyond fold-to-fold variation?** (quote the F.6 verdict). `<your answer>`

> **WRITE – what the ablation shows** (which stress type loses recall). `<your answer>`

# Section G – Evaluation (rubric G)

> Everything here is computed on the test session, once. No setting is changed after looking at these numbers.

### G.1 Fit the five models and score the test session once

In [ ]:
# G.1 fit the five models on the training rows recorded before the test session, then score the test session once
class Detector:
    """A fitted model behind one interface: score (higher = more unusual) and flag (True = anomaly)."""

    def __init__(self, name, family, params, contamination, model, columns, train_positions):
        self.name, self.family, self.params, self.contamination = name, family, params, contamination
        self.model, self.columns, self.train_positions = model, list(columns), train_positions

    def score(self, X):
        """Anomaly score: negative of score_samples for the sklearn models, the largest exceedance for the percentile rule."""
        X = X[self.columns]
        return self.model.score(X) if self.family == "rule" else -self.model.score_samples(X)

    def flag(self, X):
        """True where the model calls the row an anomaly (for the sklearn models exactly what the API's predict gives)."""
        X = X[self.columns]
        return self.model.flag(X) if self.family == "rule" else self.model.predict(X) == -1


def fit_detector(name, family, params, contamination, train_idx, columns=None):
    """Fit one final model on the given training rows (positions in model_df) and wrap it as a Detector."""
    columns = list(columns or FEATURE_NAMES)
    if family == "rule":
        positions = train_idx
        model = PercentileRule(params["percentile"]).fit(X_ALL.iloc[positions][columns])
    else:
        positions = subsample_rows(train_idx, LOF_MAX_TRAIN_ROWS if family == "lof" else IF_MAX_TRAIN_ROWS)
        model, _ = fit_pipeline(family, params, contamination, X_ALL.iloc[positions][columns])
    return Detector(name, family, params, contamination, model, columns, positions)


TEST_POS = test_fold["score_idx"]
X_test = X_ALL.iloc[TEST_POS].reset_index(drop=True)
y_test = Y_ALL[TEST_POS].astype(int)
label_test, intensity_test = LABEL_ALL[TEST_POS], INTENSITY_ALL[TEST_POS]
print(f"Test session {TEST_SESSION}: {len(y_test)} scored rows ({int((y_test == 0).sum())} normal, {int((y_test == 1).sum())} anomaly); "
      f"{len(test_fold['train_idx'])} training rows were recorded before it began.")

rule_name = f"Percentile rule (P={best_rule['percentile']:g})"
model_specs = [("Isolation Forest (default)", "iforest", IF_BASELINE, "auto"),
               ("Isolation Forest (tuned)", "iforest", IF_BEST_PARAMS, best_if["contamination"]),
               ("LOF (default)", "lof", LOF_BASELINE, "auto"),
               ("LOF (tuned)", "lof", LOF_BEST_PARAMS, best_lof["contamination"]),
               (rule_name, "rule", {"percentile": best_rule["percentile"]}, None)]
detectors = {name: fit_detector(name, family, params, contamination, test_fold["train_idx"])
             for name, family, params, contamination in model_specs}

# the one and only look at the test session
test_scores = {name: d.score(X_test) for name, d in detectors.items()}
test_flags = {name: d.flag(X_test) for name, d in detectors.items()}
test_metrics = {name: score_metrics(y_test, test_flags[name], test_scores[name], label_test) for name in detectors}
for name, d in detectors.items():
    print(f"  {name:28s} fitted on {len(d.train_positions):6d} rows | flags {test_flags[name].mean():5.1%} of the test rows")

### G.2 Prediction time and artefact size

In [ ]:
# G.2 measure prediction time per sample and the size of each fitted model
def timed(function):
    """Seconds one call of `function` takes."""
    started = time.perf_counter()
    function()
    return time.perf_counter() - started


def prediction_seconds(detector, X, repeats=5):
    """Seconds per sample for one batch of 100 rows (best of a few runs) and for one row at a time (mean of 50)."""
    batch = X.iloc[:100]
    best_batch = min(timed(lambda: (detector.flag(batch), detector.score(batch))) for _ in range(repeats))
    singles = [X.iloc[[i]] for i in range(50)]
    started = time.perf_counter()
    for row in singles:
        detector.flag(row)
        detector.score(row)
    return best_batch / len(batch), (time.perf_counter() - started) / len(singles)


def artefact_kb(detector):
    """Size of the fitted model when saved with joblib (compress=3), in kilobytes."""
    buffer = io.BytesIO()
    joblib.dump(detector.model, buffer, compress=3)
    return len(buffer.getvalue()) / 1024


runtime = {}
for name, d in detectors.items():
    per_sample_batch, per_sample_single = prediction_seconds(d, X_test)
    runtime[name] = {"prediction ms per sample (batch of 100)": per_sample_batch * 1000,
                     "prediction ms per sample (one row at a time)": per_sample_single * 1000,
                     "artefact size (KB)": artefact_kb(d)}
display(pd.DataFrame(runtime))

### G.3 One comparison table

In [ ]:
# G.3 the comparison table: every model and every metric on the test session
TYPE_INTENSITY = [(kind, level) for kind in STRESS_LABELS for level in ("moderate", "heavy")]


def recall_by_type_and_intensity(flagged, sep=" "):
    """Recall for each stress type at each intensity (8 values); sep="_" gives names that are safe as dictionary keys."""
    out = {}
    for kind, level in TYPE_INTENSITY:
        in_group = (label_test == kind) & (intensity_test == level)
        out[f"recall{sep}{kind}{sep}{level}"] = flagged[in_group].mean() if in_group.any() else np.nan
    return out


metric_labels = {"accuracy": "accuracy", "balanced_accuracy": "balanced accuracy", "precision": "precision (anomaly)",
                 "recall": "recall (anomaly)", "f1": "F1 (anomaly)", "roc_auc": "ROC-AUC",
                 "average_precision": "average precision", "fpr": "false-positive rate (normal rows)"}
comparison = pd.DataFrame({name: {**{text: test_metrics[name][key] for key, text in metric_labels.items()},
                                  **recall_by_type_and_intensity(test_flags[name]), **runtime[name]}
                           for name in detectors})
display(comparison)

### G.4 Confusion matrices

In [ ]:
# G.4 one confusion matrix per model (figure saved for the report)
fig, axes = plt.subplots(1, len(detectors), figsize=(4.1 * len(detectors), 4))
for ax, (name, flagged) in zip(axes, test_flags.items()):
    counts = confusion_matrix(y_test, flagged.astype(int), labels=[0, 1])
    shares = counts / counts.sum(axis=1, keepdims=True)
    cell_text = np.array([[f"{counts[i, j]}\n({shares[i, j]:.0%})" for j in range(2)] for i in range(2)])
    sns.heatmap(counts, annot=cell_text, fmt="", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["normal", "anomaly"], yticklabels=["normal", "anomaly"])
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

### G.5 ROC and precision-recall curves

In [ ]:
# G.5 ROC curve with ROC-AUC and precision-recall curve with average precision (figure saved for the report)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for name in detectors:
    fpr_curve, tpr_curve, _ = roc_curve(y_test, test_scores[name])
    line, = axes[0].plot(fpr_curve, tpr_curve, label=f"{name} (AUC {test_metrics[name]['roc_auc']:.3f})")
    flagged = test_flags[name]
    axes[0].scatter(flagged[y_test == 0].mean(), flagged[y_test == 1].mean(), color=line.get_color(), s=45, zorder=5)
    precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_scores[name])
    line, = axes[1].plot(recall_curve, precision_curve, label=f"{name} (AP {test_metrics[name]['average_precision']:.3f})")
    axes[1].scatter(test_metrics[name]["recall"], test_metrics[name]["precision"], color=line.get_color(), s=45, zorder=5)
axes[0].plot([0, 1], [0, 1], "k:", linewidth=1)
axes[0].set(xlabel="false-positive rate", ylabel="true-positive rate (recall)", title="ROC curve")
axes[1].axhline(y_test.mean(), color="k", linestyle=":", linewidth=1)
axes[1].set(xlabel="recall", ylabel="precision", title="Precision-recall curve (dotted = share of anomalies)")
axes[0].legend(fontsize=8, loc="lower right")
axes[1].legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.savefig(FIG_DIR / "roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### G.6 Recall for each stress type at each intensity

In [ ]:
# G.6 heatmap of recall per stress type and intensity (figure saved for the report)
recall_table = pd.DataFrame({name: recall_by_type_and_intensity(test_flags[name]) for name in detectors}).T
recall_table.columns = [column.replace("recall ", "") for column in recall_table.columns]     # keep the labels short
fig, ax = plt.subplots(figsize=(13, 3.8))
sns.heatmap(recall_table, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1, ax=ax)
ax.set_title("Recall per stress type and intensity (test session)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "recall_by_stress_type.png", dpi=150, bbox_inches="tight")
plt.show()
print("False-positive rate on normal rows:", {name: round(float(test_metrics[name]["fpr"]), 3) for name in detectors})

### G.7 Why accuracy alone is not enough

In [ ]:
# G.7 two trivial classifiers next to the real models
n_test = len(y_test)
trivial = {"always predicts normal": score_metrics(y_test, np.zeros(n_test), np.zeros(n_test)),
           "always predicts anomaly": score_metrics(y_test, np.ones(n_test), np.zeros(n_test))}
accuracy_view = pd.DataFrame({**trivial, **{name: test_metrics[name] for name in detectors}}).T
display(accuracy_view[["accuracy", "balanced_accuracy", "precision", "recall", "f1", "fpr"]])
print(f"Share of anomaly rows in the test session: {y_test.mean():.1%}  ->  a model that never flags anything scores "
      f"{1 - y_test.mean():.1%} accuracy with zero recall.")

### G.8 Which model is deployed

The ML model with the higher cross-validation F1; Isolation Forest if the gap is under 0.02.

In [ ]:
# G.8 apply the deployment rule to the cross-validation results
gap = best_lof["f1_mean"] - best_if["f1_mean"]
DEPLOYED_FAMILY = "lof" if gap >= 0.02 else "iforest"
DEPLOYED_TEST_NAME = "LOF (tuned)" if DEPLOYED_FAMILY == "lof" else "Isolation Forest (tuned)"
print(f"Cross-validation F1: Isolation Forest {best_if['f1_mean']:.3f} | LOF {best_lof['f1_mean']:.3f} | LOF minus Isolation Forest = {gap:+.3f}")
if gap >= 0.02:
    print("LOF is better by 0.02 or more -> LOF is deployed.")
elif gap > 0:
    print("LOF is better, but by less than 0.02 -> Isolation Forest is deployed (smaller, faster, stores no training rows).")
else:
    print("Isolation Forest has the higher (or an equal) cross-validation F1 -> Isolation Forest is deployed.")
print("DEPLOYED MODEL:", DEPLOYED_TEST_NAME)

best_ml_cv = max(best_if["f1_mean"], best_lof["f1_mean"])
best_ml_test = max(test_metrics["Isolation Forest (tuned)"]["f1"], test_metrics["LOF (tuned)"]["f1"])
if best_rule["f1_mean"] > best_ml_cv:
    print(f"NOTE: the percentile rule beats both ML models in cross-validation (F1 {best_rule['f1_mean']:.3f} against {best_ml_cv:.3f}). "
          "Say so in the written part and discuss what the ML models add or fail to add.")
if test_metrics[rule_name]["f1"] > best_ml_test:
    print(f"NOTE: the percentile rule also beats both tuned ML models on the test session (F1 {test_metrics[rule_name]['f1']:.3f} against {best_ml_test:.3f}).")

### G.9 Error analysis of the deployed model

In [ ]:
# G.9a score distributions of the deployed model with its threshold (figure saved for the report)
deployed_test = detectors[DEPLOYED_TEST_NAME]
score_d, flag_d = test_scores[DEPLOYED_TEST_NAME], test_flags[DEPLOYED_TEST_NAME]
threshold_d = -deployed_test.model.named_steps["model"].offset_            # flagged when the score is above this value
positive = np.clip(score_d, 1e-6, None)                                     # scores are positive; the axis is logarithmic because LOF scores have a long tail
bins = np.logspace(np.log10(positive.min()), np.log10(positive.max()), 60)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(positive[y_test == 0], bins=bins, alpha=0.7, color=LABEL_COLORS["normal"], label="normal rows")
ax.hist(positive[y_test == 1], bins=bins, alpha=0.7, color=LABEL_COLORS["cpu"], label="stress rows")
ax.axvline(threshold_d, color="black", linestyle="--", label=f"threshold ({threshold_d:.3f})")
ax.set_xscale("log")
ax.set_xlabel("anomaly score (logarithmic axis)")
ax.set_ylabel("test rows")
ax.set_title(f"{DEPLOYED_TEST_NAME}: anomaly scores on the test session")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**G.9b The mistakes in detail**

In [ ]:
# G.9b list the false positives and false negatives with their feature values
rule_fitted = detectors[rule_name].model                                    # fitted on the same training rows: gives p50 and p99
exceedance = pd.DataFrame(rule_fitted.exceedance(X_test), columns=FEATURE_NAMES)
test_view = model_df.iloc[TEST_POS].reset_index(drop=True)[["ts", "host_id", "label", "intensity"]].copy()
test_view["minutes into session"] = (test_view["ts"] - session_table.loc[TEST_SESSION, "start"]).dt.total_seconds() / 60
test_view["score"] = score_d
test_view = pd.concat([test_view, X_test[FEATURE_NAMES]], axis=1)

is_fp, is_fn, is_tp = flag_d & (y_test == 0), ~flag_d & (y_test == 1), flag_d & (y_test == 1)
print(f"{DEPLOYED_TEST_NAME}: {int(is_fp.sum())} false positives and {int(is_fn.sum())} false negatives among {len(y_test)} scored rows.")
with pd.option_context("display.float_format", "{:,.2f}".format, "display.max_columns", 30):
    print("\nFalse positives (normal rows that were flagged), highest score first:")
    display(test_view[is_fp].sort_values("score", ascending=False).head(15) if is_fp.any() else pd.DataFrame({"note": ["none"]}))
    print("False negatives (stress rows that were missed), lowest score first:")
    display(test_view[is_fn].sort_values("score").head(15) if is_fn.any() else pd.DataFrame({"note": ["none"]}))

**G.9c What the mistakes have in common**

In [ ]:
# G.9c what do the mistakes have in common?
stress_rows = test_view[y_test == 1].groupby(["label", "intensity"]).size().rename("stress rows")
missed_rows = test_view[is_fn].groupby(["label", "intensity"]).size().rename("missed")
by_group = pd.concat([stress_rows, missed_rows], axis=1).fillna(0).astype(int)
by_group["share missed"] = by_group["missed"] / by_group["stress rows"]
print("Missed stress rows by stress type and intensity:")
display(by_group)

profile = pd.DataFrame({"missed stress rows (false negatives)": exceedance[is_fn].median(),
                        "caught stress rows (true positives)": exceedance[is_tp].median(),
                        "all normal rows": exceedance[y_test == 0].median()})
print("Median exceedance per feature (0 = typical for training, 1 = at the 99th training percentile):")
display(profile)

if is_fp.any():
    print("Feature that looked most unusual in each false positive:")
    display(exceedance[is_fp].idxmax(axis=1).value_counts().rename("false positives").to_frame())
    minutes = test_view.loc[is_fp, "minutes into session"]
    print(f"False positives occurred between minute {minutes.min():.0f} and minute {minutes.max():.0f} of the session (median {minutes.median():.0f}).")

### Written part of section G

> **WRITE – strengths and limits of each approach** (numbers from G.3). `<your comparison>`

> **WRITE – why accuracy alone is not enough** (use G.7). `<your answer>`

> **WRITE – which model is deployed and why** (rule and numbers from G.8; if the percentile rule beat both ML models, say so). `<your answer>`

> **WRITE – what the errors have in common** (use G.9). `<your answer>`

> **WRITE – limits.** Four are required: no transfer to other machines; ground truth is deliberate stress; correlated samples and a single test session; accuracy depends on the protocol's class mix. `<your limits>`

# Section H – Export and deployment (rubric H)

> The deployed model is refitted on all training rows (normal and background only). The reported metrics belong to the model that never saw the test session.

### H.1 Refit on all training rows

In [ ]:
# H.1 refit the chosen configuration on ALL training rows (labels are still not used)
all_train_idx = np.flatnonzero(model_df["train_eligible"].to_numpy())
if DEPLOYED_FAMILY == "iforest":
    deployed_params, deployed_contamination = IF_BEST_PARAMS, best_if["contamination"]
else:
    deployed_params, deployed_contamination = LOF_BEST_PARAMS, best_lof["contamination"]

deployed = fit_detector("deployed", DEPLOYED_FAMILY, deployed_params, deployed_contamination, all_train_idx)
deployed_train = X_ALL.iloc[deployed.train_positions]
print("Deployed family     :", DEPLOYED_FAMILY)
print("Hyperparameters     :", {**deployed_params, "contamination": deployed_contamination})
print("Training rows       :", len(deployed.train_positions), f"(of {len(all_train_idx)} eligible rows; caps: Isolation Forest {IF_MAX_TRAIN_ROWS}, LOF {LOF_MAX_TRAIN_ROWS})")
print("Pipeline steps      :", [name for name, _ in deployed.model.steps])

### H.2 The model artefact

`model.joblib` keys: `pipeline`, `feature_names`, `model_version`, `trained_at`, `library_versions`, `feature_percentiles` (p50, p99), `test_metrics`. Only load artefacts the team made.

In [ ]:
# H.2 build the artefact dictionary and save it as model.joblib
def json_safe(mapping):
    """Plain Python numbers; NaN becomes None, so the dictionary can also be stored as JSON."""
    out = {}
    for key, value in mapping.items():
        if isinstance(value, (float, np.floating)):
            out[key] = float(value) if np.isfinite(value) else None
        elif isinstance(value, (int, np.integer)):
            out[key] = int(value)
        else:
            out[key] = value
    return out


def snake_keys(mapping):
    """Keys such as 'prediction ms per sample (batch of 100)' become 'prediction_ms_per_sample_batch_of_100'."""
    return {re.sub(r"[^0-9a-z]+", "_", str(key).lower()).strip("_"): value for key, value in mapping.items()}


trained_at = datetime.now(timezone.utc)
model_version =f"{'if' if DEPLOYED_FAMILY == 'iforest' else 'lof'}-{trained_at:%Y-%m-%d}"
artefact = {
    "pipeline": deployed.model,
    "feature_names": list(FEATURE_NAMES),
    "model_version": model_version,
    "trained_at": trained_at.isoformat(),
    "library_versions": {key: LIBRARY_VERSIONS[key] for key in ("python", "numpy", "pandas", "scikit-learn", "joblib")},
    "feature_percentiles": {"p50": {k: float(v) for k, v in deployed_train.quantile(0.5).items()},
                            "p99": {k: float(v) for k, v in deployed_train.quantile(0.99).items()}},
    "test_metrics": json_safe({**test_metrics[DEPLOYED_TEST_NAME], **recall_by_type_and_intensity(test_flags[DEPLOYED_TEST_NAME], sep="_"),
                               "test_session": TEST_SESSION, "test_rows": int(len(y_test))}),
    # extra keys for traceability (the API can ignore them)
    "model_family": DEPLOYED_FAMILY,
    "hyperparameters": json_safe({**deployed_params, "contamination": deployed_contamination}),
    "window_seconds": WINDOW_SECONDS,
    "exceedance_eps": EXCEEDANCE_EPS,
}
joblib.dump(artefact, "model.joblib", compress=3)
print("Saved model.joblib:", f"{Path('model.joblib').stat().st_size / 1024:,.0f} KB", "| model_version:", model_version)
print("Keys:", list(artefact))
print("Test metrics stored:", {k: round(v, 3) for k, v in artefact["test_metrics"].items() if isinstance(v, float)})

pins = [("scikit-learn", "scikit-learn"), ("numpy", "numpy"), ("pandas", "pandas"), ("joblib", "joblib")]
requirements_pins = "\n".join(f"{package}=={LIBRARY_VERSIONS[key]}" for package, key in pins)
Path("requirements_pins.txt").write_text(requirements_pins + f"\n# trained with Python {LIBRARY_VERSIONS['python']}\n")
print("\nPin these in backend/requirements.txt:\n" + requirements_pins)

### H.3 Check the saved file as the API will use it

In [ ]:
# H.3 reload the saved artefact and compare it with the model in memory
loaded = joblib.load("model.joblib")
assert loaded["feature_names"] == FEATURE_NAMES, "feature order differs from features.py"
sample = X_ALL.iloc[TEST_POS[:500]][FEATURE_NAMES]
assert (loaded["pipeline"].predict(sample) == deployed.model.predict(sample)).all(), "reloaded model predicts differently"
assert np.allclose(loaded["pipeline"].score_samples(sample), deployed.model.score_samples(sample)), "reloaded scores differ"
for step_name, step in loaded["pipeline"].steps:
    assert type(step).__module__.startswith("sklearn."), f"{step_name} is not a standard scikit-learn class"
assert loaded["library_versions"]["scikit-learn"] == sklearn.__version__ and loaded["library_versions"]["numpy"] == np.__version__
print("Reloaded artefact gives identical predictions and scores; steps:", [type(s).__name__ for _, s in loaded["pipeline"].steps])
print("model_version:", loaded["model_version"], "| trained_at:", loaded["trained_at"])
print("p99 of cpu_percent in the training rows:", round(loaded["feature_percentiles"]["p99"]["cpu_percent"], 2))

### H.4 Check that live scoring matches training

In [ ]:
# H.4 simulate the API: rebuild the features of random small requests and compare them with the training-time features
rng = np.random.RandomState(RANDOM_STATE)
parity_host = sys_clean["host_id"].value_counts().idxmax()
host_rows = sys_clean[sys_clean["host_id"] == parity_host].sort_values("ts", kind="mergesort")
largest_difference, compared = 0.0, 0
for _ in range(60):
    first = rng.randint(0, len(host_rows) - 12)
    new_rows = host_rows.iloc[first:first + rng.randint(1, 13)]
    window = host_rows[(host_rows["ts"] >= new_rows["ts"].iloc[0] - pd.Timedelta(seconds=WINDOW_SECONDS)) & (host_rows["ts"] <= new_rows["ts"].iloc[-1])]
    served = build_features(window).loc[new_rows.index]          # what the API would compute
    trained = feat_all.loc[new_rows.index]                       # what the notebook used for training
    assert (served.isna().to_numpy() == trained.isna().to_numpy()).all(), "missing-value pattern differs between serving and training"
    difference = np.abs(served.to_numpy(dtype=float) - trained.to_numpy(dtype=float))
    largest_difference = max(largest_difference, float(np.nanmax(difference)) if np.isfinite(difference).any() else 0.0)
    compared += len(new_rows)
assert largest_difference < 1e-9, f"serving features differ from training features by {largest_difference}"
print(f"{compared} samples in 60 simulated requests (host {parity_host}): largest difference between serving and training features = "
      f"{largest_difference:.2e} (the limit is 1e-9; 0 means bit-for-bit identical)")

### H.5 Health check of the deployed API

Set `API_URL` in the run settings. The answer must show the exported `model_version`.

In [ ]:
# H.5 call the deployed API's health endpoint and compare the model version with the exported artefact
import urllib.error
import urllib.request

if not API_URL:
    print("API_URL is empty. Set it in the run-settings cell (for example 'https://<your-api>.onrender.com') and rerun this cell.")
else:
    health_url = API_URL.rstrip("/") + "/v1/health"
    try:
        with urllib.request.urlopen(health_url, timeout=90) as response:
            health = json.loads(response.read().decode("utf-8"))
        print(json.dumps(health, indent=2))
        print("Model version in model.joblib:", model_version)
        print("The deployed API reports the same version:", model_version in json.dumps(health))
    except (urllib.error.URLError, TimeoutError, ValueError) as error:
        print("Could not read", health_url, "->", error)

### H.6 Collect the outputs

Save this notebook as `ml/notebook.ipynb`; commit `model.joblib` and `features.py` as `ml/model.joblib` and `ml/features.py`.

In [ ]:
# H.6 save the tables, zip the outputs and (in Colab) download the ZIP
cv_all = pd.concat([cv_if_base.assign(stage="default"), cv_if.assign(stage="grid"), cv_lof_base.assign(stage="default"),
                    cv_lof.assign(stage="grid"), cv_rule_base.assign(stage="default"), cv_rule.assign(stage="grid")], ignore_index=True)
cv_all.to_csv("cv_results.csv", index=False)
comparison.to_csv("comparison_table.csv")
all_test_metrics = {name: json_safe({**test_metrics[name], **recall_by_type_and_intensity(test_flags[name], sep="_"), **snake_keys(runtime[name])})
                    for name in detectors}
Path("test_metrics.json").write_text(json.dumps({"test_session": TEST_SESSION, "deployed_model": DEPLOYED_TEST_NAME,
                                                 "model_version": model_version, "models": all_test_metrics}, indent=2))

output_files = ["model.joblib", "features.py", "requirements_pins.txt", "test_metrics.json", "comparison_table.csv", "cv_results.csv"]
output_files += [str(p) for p in sorted(FIG_DIR.glob("*.png"))]
with zipfile.ZipFile("kali_anomaly_ml_outputs.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for file_name in output_files:
        bundle.write(file_name)
print("Zipped", len(output_files), "files into kali_anomaly_ml_outputs.zip:")
for file_name in output_files:
    print("  ", file_name)

try:
    from google.colab import files
    files.download("kali_anomaly_ml_outputs.zip")        # opens the browser's download dialog
except ImportError:
    print("\nNot running in Colab: the ZIP is in the working folder.")

### H.7 Screenshots

Put `agent.png`, `api.png` and `dashboard.png` in a folder called `screenshots`.

In [ ]:
# H.7 display the screenshots when the files exist
from IPython.display import Image

SCREENSHOTS = {"Monitoring agent: service running and sending samples": "screenshots/agent.png",
               "Cloud API: health check or an ingest result": "screenshots/api.png",
               "Dashboard: live metrics and an anomaly event with likely contributors": "screenshots/dashboard.png"}
for caption, path in SCREENSHOTS.items():
    if Path(path).exists():
        print(caption)
        display(Image(filename=path))
    else:
        print("MISSING:", path, "-", caption)

> **WRITE – deployment evidence** (one or two sentences per screenshot, and the public URLs). `<your text>`

# Conclusion

> **WRITE – conclusion** (half a page): what was built, what the test session showed, what worked and what did not, limits, next steps. `<your conclusion>`

# References

- Liu, Ting, Zhou (2008). *Isolation Forest.* IEEE ICDM.
- Breunig, Kriegel, Ng, Sander (2000). *LOF: Identifying Density-Based Local Outliers.* ACM SIGMOD.
- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python.* JMLR 12.
- Project technical specification v1.0 (17 Nov 2026).